In [2]:
# ============================================================
# BLOCK 1: Environment Setup
# CRATTT Clean Implementation
# Dada Victor Damilare | MRES7015 | University of Greater Manchester
# ============================================================

# --- 1.1 Install Required Libraries ---
!pip install imagecorruptions -q
!pip install wandb -q
!pip install pycocotools -q
!pip install ultralytics -q

# --- 1.2 Core Imports ---
import os
import sys
import random
import json
import glob
import numpy as np
import torch
import skimage
import skimage.filters
import imagecorruptions.corruptions as cor_mod
import imagecorruptions
import wandb
from kaggle_secrets import UserSecretsClient
from huggingface_hub import login

# --- 1.3 Reproducibility: Fix All Seeds ---
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print(f"✅ Seeds fixed: {SEED}")

# --- 1.4 Surgical Patch: NumPy 2.0 + scikit-image compatibility ---
# imagecorruptions uses the deprecated 'multichannel' kwarg removed in
# scikit-image 0.19+. This patch redirects it to the new 'channel_axis'.
if not hasattr(skimage.filters.gaussian, '_is_patched'):
    _real_gaussian = skimage.filters.gaussian

    def _patched_gaussian(*args, **kwargs):
        if 'multichannel' in kwargs:
            val = kwargs.pop('multichannel')
            kwargs['channel_axis'] = -1 if val else None
        return _real_gaussian(*args, **kwargs)

    _patched_gaussian._is_patched = True
    skimage.filters.gaussian = _patched_gaussian
    cor_mod.gaussian = _patched_gaussian
    print("✅ scikit-image patch applied")
else:
    print("✅ scikit-image patch already active")

# --- 1.5 Log Library Versions for Reproducibility ---
print("\n--- Library Versions ---")
print(f"Python:            {sys.version.split()[0]}")
print(f"PyTorch:           {torch.__version__}")
print(f"NumPy:             {np.__version__}")
print(f"scikit-image:      {skimage.__version__}")

import importlib.metadata
try:
    ic_version = importlib.metadata.version("imagecorruptions")
except importlib.metadata.PackageNotFoundError:
    ic_version = "installed (version unknown)"
print(f"imagecorruptions:  {ic_version}")

# --- 1.6 Device ---
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✅ Device: {device}")
if device.type == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

# --- 1.7 Secure API Connections ---
user_secrets = UserSecretsClient()
print("\n--- API Connections ---")

try:
    wb_key = user_secrets.get_secret("WANDB_API_KEY")
    wandb.login(key=wb_key, relogin=True)
    print("✅ W&B: Connected")
except Exception as e:
    print(f"⚠️  W&B: Not connected — {e}")

try:
    hf_token = user_secrets.get_secret("HF_TOKEN")
    login(token=hf_token)
    print("✅ HuggingFace: Connected")
except Exception as e:
    print(f"❌ HuggingFace: Failed — {e}")

# --- 1.8 Create Output Directories ---
DIRS = {
    "results":  "/kaggle/working/results",
    "figures":  "/kaggle/working/figures",
    "checkpoints": "/kaggle/working/checkpoints",
    "tables":   "/kaggle/working/tables"
}
for name, path in DIRS.items():
    os.makedirs(path, exist_ok=True)

print(f"\n✅ Output directories created:")
for name, path in DIRS.items():
    print(f"   {name}: {path}")

print("\n" + "="*50)
print("BLOCK 1 COMPLETE — Environment ready")
print("="*50)

✅ Seeds fixed: 42
✅ scikit-image patch already active

--- Library Versions ---
Python:            3.12.13
PyTorch:           2.10.0+cu128
NumPy:             2.4.6
scikit-image:      0.25.2
imagecorruptions:  1.1.2

✅ Device: cuda
   GPU: Tesla T4
   VRAM: 15.6 GB

--- API Connections ---


wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: vd1res (vd1res-university-of-bolton) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


✅ W&B: Connected
✅ HuggingFace: Connected

✅ Output directories created:
   results: /kaggle/working/results
   figures: /kaggle/working/figures
   checkpoints: /kaggle/working/checkpoints
   tables: /kaggle/working/tables

BLOCK 1 COMPLETE — Environment ready


In [3]:
# ============================================================
# BLOCK 2: Model Loading
# Loads GroundingDINO, YOLO-World, and CLIP Oracle once.
# Never reload models in subsequent blocks.
# ============================================================

import torch
from transformers import AutoProcessor, AutoModelForZeroShotObjectDetection
from transformers import CLIPProcessor, CLIPModel
from ultralytics import YOLOWorld

print("Loading models to device:", device)
print("="*50)

# --- 2.1 GroundingDINO ---
print("\n[1/3] Loading GroundingDINO-tiny...")
DINO_MODEL_ID = "IDEA-Research/grounding-dino-tiny"

dino_processor = AutoProcessor.from_pretrained(
    DINO_MODEL_ID, token=hf_token
)
dino_model = AutoModelForZeroShotObjectDetection.from_pretrained(
    DINO_MODEL_ID, token=hf_token
).to(device)
dino_model.eval()

dino_params = sum(p.numel() for p in dino_model.parameters())
print(f"✅ GroundingDINO-tiny loaded")
print(f"   Parameters: {dino_params:,}")
print(f"   VRAM used so far: "
      f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

# --- 2.2 YOLO-World ---
print("\n[2/3] Loading YOLO-World-large...")

# CPU handoff pattern: set classes on CPU to avoid CUDA text-encoding bug
yolo_model = YOLOWorld('yolov8l-world.pt')
yolo_model.to('cpu')

# We will set the full class list in Block 3 once COCO_MAP is defined
# For now just verify it loads
print(f"✅ YOLO-World-large loaded")
print(f"   VRAM used so far: "
      f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

# --- 2.3 CLIP Oracle ---
print("\n[3/3] Loading CLIP Oracle (ViT-B/32)...")
CLIP_MODEL_ID = "openai/clip-vit-base-patch32"

clip_processor = CLIPProcessor.from_pretrained(CLIP_MODEL_ID)
clip_model = CLIPModel.from_pretrained(CLIP_MODEL_ID).to(device)
clip_model.eval()

clip_params = sum(p.numel() for p in clip_model.parameters())
print(f"✅ CLIP Oracle loaded")
print(f"   Parameters: {clip_params:,}")
print(f"   VRAM used so far: "
      f"{torch.cuda.memory_allocated() / 1e9:.2f} GB")

# --- 2.4 VRAM Summary ---
vram_allocated = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_free = vram_total - vram_allocated

print("\n--- VRAM Summary ---")
print(f"Allocated: {vram_allocated:.2f} GB")
print(f"Free:      {vram_free:.2f} GB")
print(f"Total:     {vram_total:.2f} GB")

if vram_free < 4.0:
    print("⚠️  WARNING: Less than 4GB free. "
          "Consider reducing NUM_IMAGES in Block 4.")
else:
    print("✅ VRAM headroom is sufficient for inference.")

print("\n" + "="*50)
print("BLOCK 2 COMPLETE — All models loaded")
print("="*50)

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Loading models to device: cuda

[1/3] Loading GroundingDINO-tiny...


preprocessor_config.json:   0%|          | 0.00/457 [00:00<?, ?B/s]

The image processor of type `GroundingDinoImageProcessor` is now loaded as a fast processor by default, even if the model checkpoint was saved with a slow processor. This is a breaking change and may produce slightly different outputs. To continue using the slow processor, instantiate this class with `use_fast=False`. 


config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

CLIPModel LOAD REPORT from: openai/clip-vit-base-patch32
Key                                  | Status     |  | 
-------------------------------------+------------+--+-
vision_model.embeddings.position_ids | UNEXPECTED |  | 
text_model.embeddings.position_ids   | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ CLIP Oracle loaded
   Parameters: 151,277,313
   VRAM used so far: 1.30 GB

--- VRAM Summary ---
Allocated: 1.30 GB
Free:      14.34 GB
Total:     15.64 GB
✅ VRAM headroom is sufficient for inference.

BLOCK 2 COMPLETE — All models loaded


model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

In [4]:
# ============================================================
# BLOCK 3: Global Constants
# Single source of truth for all experimental parameters.
# Never hardcode these values in subsequent blocks.
# ============================================================

import torch
import torch.nn.functional as F

# --- 3.1 COCO Category Mapping ---
# Verified against official COCO 2017 annotation file
COCO_MAP = {
    'person': 1, 'bicycle': 2, 'car': 3, 'motorcycle': 4,
    'airplane': 5, 'bus': 6, 'train': 7, 'truck': 8, 'boat': 9,
    'traffic light': 10, 'fire hydrant': 11, 'stop sign': 13,
    'parking meter': 14, 'bench': 15, 'bird': 16, 'cat': 17,
    'dog': 18, 'horse': 19, 'sheep': 20, 'cow': 21,
    'elephant': 22, 'bear': 23, 'zebra': 24, 'giraffe': 25,
    'backpack': 27, 'umbrella': 28, 'handbag': 31, 'tie': 32,
    'suitcase': 33, 'frisbee': 34, 'skis': 35, 'snowboard': 36,
    'sports ball': 37, 'kite': 38, 'baseball bat': 39,
    'baseball glove': 40, 'skateboard': 41, 'surfboard': 42,
    'tennis racket': 43, 'bottle': 44, 'wine glass': 46,
    'cup': 47, 'fork': 48, 'knife': 49, 'spoon': 50, 'bowl': 51,
    'banana': 52, 'apple': 53, 'sandwich': 54, 'orange': 55,
    'broccoli': 56, 'carrot': 57, 'hot dog': 58, 'pizza': 59,
    'donut': 60, 'cake': 61, 'chair': 62, 'couch': 63,
    'potted plant': 64, 'bed': 65, 'dining table': 67,
    'toilet': 70, 'tv': 72, 'laptop': 73, 'mouse': 74,
    'remote': 75, 'keyboard': 76, 'cell phone': 77,
    'microwave': 78, 'oven': 79, 'toaster': 80, 'sink': 81,
    'refrigerator': 82, 'book': 84, 'clock': 85, 'vase': 86,
    'scissors': 87, 'teddy bear': 88, 'hair drier': 89,
    'toothbrush': 90
}

COCO_CLASSES = list(COCO_MAP.keys())  # 80 classes
print(f"✅ COCO_MAP loaded: {len(COCO_MAP)} categories")

# --- 3.2 Verify COCO_MAP Against Ground Truth ---
# This catches any ID mismatches before they silently corrupt mAP scores
from pycocotools.coco import COCO

ANN_PATH = ("/kaggle/input/datasets/awsaf49/"
            "coco-2017-dataset/coco2017/annotations/"
            "instances_val2017.json")
IMAGE_DIR = ("/kaggle/input/datasets/awsaf49/"
             "coco-2017-dataset/coco2017/val2017")

coco_gt = COCO(ANN_PATH)
official_cats = {
    cat['name']: cat['id']
    for cat in coco_gt.loadCats(coco_gt.getCatIds())
}

mismatches = []
for name, our_id in COCO_MAP.items():
    official_id = official_cats.get(name)
    if official_id is None:
        mismatches.append(f"  NOT FOUND in COCO: '{name}'")
    elif official_id != our_id:
        mismatches.append(
            f"  MISMATCH: '{name}' → ours={our_id}, "
            f"official={official_id}"
        )

if mismatches:
    print("⚠️  COCO_MAP mismatches detected:")
    for m in mismatches:
        print(m)
else:
    print("✅ COCO_MAP verified: all 80 IDs match official annotations")

# --- 3.3 Text Prompts ---
# GroundingDINO expects dot-separated class names
DINO_TEXT_PROMPT = " . ".join(COCO_CLASSES) + " ."

# YOLO-World expects plain class names
# Set classes here using the CPU handoff pattern
yolo_model.to('cpu')
yolo_model.set_classes(COCO_CLASSES)
yolo_model.to(device)
print("✅ YOLO-World classes set: 80 COCO categories")


# --- 3.4 CLIP Per-Class Text Embeddings ---
print("\nGenerating per-class CLIP text embeddings...")

with torch.no_grad():
    embeddings = []
    
    for class_name in COCO_CLASSES:
        inputs = clip_processor(
            text=[class_name],
            return_tensors="pt",
            padding=True
        ).to(device)
        
        # Bypass get_text_features entirely
        # Call text_model directly, extract last hidden state,
        # then apply projection manually
        text_out = clip_model.text_model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"]
        )
        
        # Pooled output is the [EOS] token representation
        # Shape: [1, hidden_dim] = [1, 512]
        pooled = text_out.pooler_output
        
        # Apply the learned projection to get final embedding
        projected = clip_model.text_projection(pooled)
        embeddings.append(projected)
    
    clip_text_features = torch.cat(embeddings, dim=0)
    clip_text_features = F.normalize(clip_text_features, p=2, dim=-1)

print(f"✅ CLIP text embeddings: {clip_text_features.shape}")
print(f"   Expected: torch.Size([80, 512])")

person_idx = COCO_CLASSES.index('person')
car_idx    = COCO_CLASSES.index('car')
chair_idx  = COCO_CLASSES.index('chair')
dog_idx    = COCO_CLASSES.index('dog')

sim_person_car  = (clip_text_features[person_idx] @ clip_text_features[car_idx]).item()
sim_person_dog  = (clip_text_features[person_idx] @ clip_text_features[dog_idx]).item()
sim_car_chair   = (clip_text_features[car_idx] @ clip_text_features[chair_idx]).item()

print(f"\n   Similarity checks (all must be < 0.98):")
print(f"   person vs car:   {sim_person_car:.4f}")
print(f"   person vs dog:   {sim_person_dog:.4f}")
print(f"   car vs chair:    {sim_car_chair:.4f}")

all_distinct = all(
    s < 0.98 for s in [sim_person_car, sim_person_dog, sim_car_chair]
)
print(f"\n   {'✅ All embeddings are distinct' if all_distinct else '❌ Still identical — check CLIP model'}")


# --- 3.5 ImageNet-C Corruption Protocol ---
CORRUPTION_CATEGORIES = {
    "Noise":   ["gaussian_noise", "shot_noise", "impulse_noise"],
    "Blur":    ["defocus_blur", "glass_blur", "motion_blur", "zoom_blur"],
    "Weather": ["snow", "frost", "fog", "brightness"],
    "Digital": ["contrast", "elastic_transform",
                "pixelate", "jpeg_compression"]
}
ALL_CORRUPTIONS = [c for cats in CORRUPTION_CATEGORIES.values()
                   for c in cats]
SEVERITIES = [1, 2, 3, 4, 5]

print(f"\n✅ ImageNet-C protocol:")
print(f"   Categories: {list(CORRUPTION_CATEGORIES.keys())}")
print(f"   Total corruptions: {len(ALL_CORRUPTIONS)}")
print(f"   Severities: {SEVERITIES}")

# --- 3.6 CRATTT Hyperparameters ---
# Fixed values from Chapter 3 pilot analysis
# Do not change these for primary results
# Use ablation blocks for sensitivity analysis
CRATTT_PARAMS = {
    "alpha":          0.4,   # DINO weight in Rjoint
    "beta":           0.6,   # Oracle weight in Rjoint
    "tau":            0.25,  # Fixed verification threshold
    "dino_text_thr":  0.12,  # GroundingDINO text threshold (permissive)
    "yolo_conf":      0.12,  # YOLO confidence (matched to DINO)
    "max_regions":    15,    # BARON max region crops
    "region_size":    (224, 224),  # CLIP input size
}
print(f"\n✅ CRATTT hyperparameters locked:")
for k, v in CRATTT_PARAMS.items():
    print(f"   {k}: {v}")

# --- 3.7 Evaluation Settings ---
EVAL_PARAMS = {
    "num_images":   20,
    "num_pilot":    5,
    "save_dir":     DIRS["results"],
    "fig_dir":      DIRS["figures"],
    "table_dir":    DIRS["tables"],
    "ckpt_dir":     DIRS["checkpoints"],
}
print(f"\n✅ Evaluation parameters:")
for k, v in EVAL_PARAMS.items():
    print(f"   {k}: {v}")

print("\n" + "="*50)
print("BLOCK 3 COMPLETE — Constants defined")
print("="*50)

✅ COCO_MAP loaded: 80 categories
loading annotations into memory...
Done (t=0.89s)
creating index...
index created!
✅ COCO_MAP verified: all 80 IDs match official annotations
requirements: Ultralytics requirement ['git+https://github.com/ultralytics/CLIP.git'] not found, attempting AutoUpdate...
Using Python 3.12.13 environment at: /usr
Resolved 36 packages in 1.28s
Prepared 2 packages in 2.75s
Installed 2 packages in 2ms
 + clip==1.0 (from git+https://github.com/ultralytics/CLIP.git@1d071f91693d6ea610252ed16a9dfc60cc0a0825)
 + ftfy==6.3.1

requirements: AutoUpdate success ✅ 4.4s
WARNING ⚠️ requirements: Restart runtime or rerun command for updates to take effect



100%|███████████████████████████████████████| 338M/338M [00:04<00:00, 74.3MiB/s]


✅ YOLO-World classes set: 80 COCO categories

Generating per-class CLIP text embeddings...
✅ CLIP text embeddings: torch.Size([80, 512])
   Expected: torch.Size([80, 512])

   Similarity checks (all must be < 0.98):
   person vs car:   0.8450
   person vs dog:   0.8424
   car vs chair:    0.8068

   ✅ All embeddings are distinct

✅ ImageNet-C protocol:
   Categories: ['Noise', 'Blur', 'Weather', 'Digital']
   Total corruptions: 15
   Severities: [1, 2, 3, 4, 5]

✅ CRATTT hyperparameters locked:
   alpha: 0.4
   beta: 0.6
   tau: 0.25
   dino_text_thr: 0.12
   yolo_conf: 0.12
   max_regions: 15
   region_size: (224, 224)

✅ Evaluation parameters:
   num_images: 20
   num_pilot: 5
   save_dir: /kaggle/working/results
   fig_dir: /kaggle/working/figures
   table_dir: /kaggle/working/tables
   ckpt_dir: /kaggle/working/checkpoints

BLOCK 3 COMPLETE — Constants defined


In [5]:
# ============================================================
# BLOCK 4: Data Loading & COCO Setup
# Loads 20 COCO validation images into memory once.
# Verifies ground truth annotations are accessible.
# ============================================================

import os
import glob
import numpy as np
from PIL import Image
from tqdm.notebook import tqdm

# --- 4.1 Load Image File Paths ---
image_files = sorted(glob.glob(os.path.join(IMAGE_DIR, "*.jpg")))

assert len(image_files) > 0, \
    f"No images found at {IMAGE_DIR} — check dataset path"

# Take the first NUM_IMAGES for evaluation
image_files = image_files[:EVAL_PARAMS["num_images"]]
print(f"✅ Found {len(image_files)} images for evaluation")

# --- 4.2 Build Image ID Map ---
# COCO image IDs are encoded in the filename e.g. 000000000139.jpg → 139
img_id_map = {
    os.path.basename(f): int(os.path.basename(f).split('.')[0])
    for f in image_files
}
coco_img_ids = list(img_id_map.values())
print(f"✅ Image ID map built: {len(img_id_map)} entries")
print(f"   First 3 IDs: {coco_img_ids[:3]}")

# --- 4.3 Pre-load Images Into Memory ---
# Avoids repeated disk reads during the benchmark loops
print(f"\nPre-loading {len(image_files)} images into memory...")
loaded_images = {}

for img_path in tqdm(image_files, desc="Loading"):
    img_array = np.array(Image.open(img_path).convert("RGB"))
    loaded_images[img_path] = img_array

# Memory estimate
sample_shape = next(iter(loaded_images.values())).shape
total_mb = sum(
    img.nbytes for img in loaded_images.values()
) / 1e6

print(f"✅ All images loaded")
print(f"   Sample shape: {sample_shape}")
print(f"   Total memory: {total_mb:.1f} MB")

# --- 4.4 Verify COCO Annotations ---
# Check that ground truth boxes exist for our image IDs
print(f"\nVerifying COCO annotations...")
missing_annotations = []

for img_id in coco_img_ids:
    ann_ids = coco_gt.getAnnIds(imgIds=img_id)
    if len(ann_ids) == 0:
        missing_annotations.append(img_id)

if missing_annotations:
    print(f"⚠️  {len(missing_annotations)} images have no annotations: "
          f"{missing_annotations}")
else:
    print(f"✅ All {len(coco_img_ids)} images have ground truth annotations")

# --- 4.5 Annotation Statistics ---
# Useful context for interpreting mAP results
total_gt_boxes = 0
category_counts = {}

for img_id in coco_img_ids:
    ann_ids = coco_gt.getAnnIds(imgIds=img_id)
    anns = coco_gt.loadAnns(ann_ids)
    total_gt_boxes += len(anns)
    
    for ann in anns:
        cat_name = coco_gt.loadCats(ann['category_id'])[0]['name']
        category_counts[cat_name] = category_counts.get(cat_name, 0) + 1

# Top 10 most frequent categories in our evaluation set
top_cats = sorted(category_counts.items(), 
                  key=lambda x: x[1], reverse=True)[:10]

print(f"\n--- Ground Truth Statistics ---")
print(f"Total GT boxes across {len(image_files)} images: {total_gt_boxes}")
print(f"Mean GT boxes per image: {total_gt_boxes/len(image_files):.1f}")
print(f"\nTop 10 categories in evaluation set:")
for cat, count in top_cats:
    print(f"   {cat:<20} {count:>4} instances")

# --- 4.6 Save Dataset Manifest ---
import json

manifest = {
    "num_images": len(image_files),
    "image_ids": coco_img_ids,
    "total_gt_boxes": total_gt_boxes,
    "mean_gt_per_image": round(total_gt_boxes / len(image_files), 2),
    "top_categories": dict(top_cats)
}

manifest_path = os.path.join(EVAL_PARAMS["save_dir"], "dataset_manifest.json")
with open(manifest_path, "w") as f:
    json.dump(manifest, f, indent=2)

print(f"\n✅ Dataset manifest saved: {manifest_path}")

print("\n" + "="*50)
print("BLOCK 4 COMPLETE — Data loaded and verified")
print("="*50)

✅ Found 20 images for evaluation
✅ Image ID map built: 20 entries
   First 3 IDs: [139, 285, 632]

Pre-loading 20 images into memory...


Loading:   0%|          | 0/20 [00:00<?, ?it/s]

✅ All images loaded
   Sample shape: (426, 640, 3)
   Total memory: 16.6 MB

Verifying COCO annotations...
✅ All 20 images have ground truth annotations

--- Ground Truth Statistics ---
Total GT boxes across 20 images: 143
Mean GT boxes per image: 7.2

Top 10 categories in evaluation set:
   person                 54 instances
   book                   16 instances
   car                     8 instances
   chair                   6 instances
   vase                    4 instances
   potted plant            3 instances
   tv                      3 instances
   teddy bear              3 instances
   handbag                 3 instances
   backpack                3 instances

✅ Dataset manifest saved: /kaggle/working/results/dataset_manifest.json

BLOCK 4 COMPLETE — Data loaded and verified


In [ ]:
# ============================================================
# BLOCK 5: Clean Baseline mAP
# Evaluates GroundingDINO and YOLO-World on uncorrupted images.
# This is the anchor point for all mCE calculations.
# ============================================================

import torch
import numpy as np
import json
import os
from tqdm.notebook import tqdm
from pycocotools.cocoeval import COCOeval

# --- 5.1 COCO Format Conversion Helpers ---
def dino_to_coco_format(results, img_id):
    """
    Converts GroundingDINO output to COCO evaluation format.
    Returns list of dicts with image_id, category_id, bbox, score.
    """
    coco_preds = []
    labels = results.get("text_labels", results.get("labels", []))

    for box, score, label in zip(
        results["boxes"], results["scores"], labels
    ):
        if not isinstance(label, str):
            continue

        clean_label = label.lower().replace(".", "").strip()
        category_id = COCO_MAP.get(clean_label)

        if category_id is None:
            continue

        box = box.tolist()
        coco_preds.append({
            "image_id":   img_id,
            "category_id": category_id,
            "bbox": [
                box[0],
                box[1],
                box[2] - box[0],  # width
                box[3] - box[1]   # height
            ],
            "score": float(score)
        })

    return coco_preds


def yolo_to_coco_format(results, img_id):
    """
    Converts YOLO-World output to COCO evaluation format.
    """
    coco_preds = []
    names = yolo_model.names

    for box in results.boxes:
        raw_name  = names[int(box.cls[0])]
        clean_name = raw_name.lower().replace("_", " ").strip()
        category_id = COCO_MAP.get(clean_name)

        if category_id is None:
            continue

        b = box.xyxy[0].tolist()
        coco_preds.append({
            "image_id":    img_id,
            "category_id": category_id,
            "bbox": [
                b[0],
                b[1],
                b[2] - b[0],
                b[3] - b[1]
            ],
            "score": float(box.conf)
        })

    return coco_preds


def compute_map(predictions, coco_gt_obj, img_ids):
    """
    Computes mAP@0.50:0.95 using pycocotools.
    Returns (mAP, status_string).
    """
    if not predictions:
        return 0.0, "no_predictions"

    try:
        import sys
        old_stdout = sys.stdout
        sys.stdout = open(os.devnull, 'w')

        dt  = coco_gt_obj.loadRes(predictions)
        ev  = COCOeval(coco_gt_obj, dt, 'bbox')
        ev.params.imgIds = img_ids
        ev.evaluate()
        ev.accumulate()
        ev.summarize()

        sys.stdout.close()
        sys.stdout = old_stdout

        return ev.stats[0], "ok"

    except Exception as e:
        # Always restore stdout even on failure
        try:
            sys.stdout.close()
        except:
            pass
        sys.stdout = old_stdout
        return 0.0, f"error: {e}"


# --- 5.2 Run Clean Baseline Inference ---
print("⚓ Running clean baseline inference...")
print(f"   Images: {len(image_files)}")
print(f"   DINO text threshold: {CRATTT_PARAMS['dino_text_thr']}")
print(f"   YOLO conf threshold: {CRATTT_PARAMS['yolo_conf']}")

dino_clean_preds = []
yolo_clean_preds = []

for img_path in tqdm(image_files, desc="Clean baseline"):
    img_id  = img_id_map[os.path.basename(img_path)]
    raw_img = loaded_images[img_path]

    # GroundingDINO inference
    inputs = dino_processor(
        images=raw_img,
        text=DINO_TEXT_PROMPT,
        return_tensors="pt"
    ).to(device)

    with torch.no_grad():
        outputs = dino_model(**inputs)

    res = dino_processor.post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        target_sizes=[raw_img.shape[:2]],
        text_threshold=CRATTT_PARAMS["dino_text_thr"]
    )[0]

    dino_clean_preds.extend(dino_to_coco_format(res, img_id))

    # YOLO-World inference
    y_res = yolo_model.predict(
        raw_img,
        conf=CRATTT_PARAMS["yolo_conf"],
        verbose=False
    )[0]

    yolo_clean_preds.extend(yolo_to_coco_format(y_res, img_id))

print(f"\n   DINO raw predictions:  {len(dino_clean_preds)}")
print(f"   YOLO raw predictions:  {len(yolo_clean_preds)}")

# --- 5.3 Compute Clean mAP ---
print("\nComputing mAP@0.50:0.95...")

map_clean_dino, status_dino = compute_map(
    dino_clean_preds, coco_gt, coco_img_ids
)
map_clean_yolo, status_yolo = compute_map(
    yolo_clean_preds, coco_gt, coco_img_ids
)

print(f"\n{'='*40}")
print(f"CLEAN BASELINE RESULTS")
print(f"{'='*40}")
print(f"GroundingDINO mAP@50:95 : {map_clean_dino:.4f}  [{status_dino}]")
print(f"YOLO-World    mAP@50:95 : {map_clean_yolo:.4f}  [{status_yolo}]")
print(f"{'='*40}")

if status_dino != "ok" or status_yolo != "ok":
    print("⚠️  One or more evaluations failed — check status strings above")

# --- 5.4 Persist Clean Baseline ---
clean_baseline = {
    "map_clean_dino":    float(map_clean_dino),
    "map_clean_yolo":    float(map_clean_yolo),
    "status_dino":       status_dino,
    "status_yolo":       status_yolo,
    "n_images":          len(image_files),
    "dino_text_thr":     CRATTT_PARAMS["dino_text_thr"],
    "yolo_conf":         CRATTT_PARAMS["yolo_conf"],
    "dino_n_preds":      len(dino_clean_preds),
    "yolo_n_preds":      len(yolo_clean_preds)
}

baseline_path = os.path.join(
    EVAL_PARAMS["save_dir"], "clean_baseline.json"
)
with open(baseline_path, "w") as f:
    json.dump(clean_baseline, f, indent=2)

print(f"\n✅ Clean baseline saved: {baseline_path}")
print("\n" + "="*50)
print("BLOCK 5 COMPLETE — Clean baseline established")
print("="*50)

In [ ]:
# ============================================================
# PRE-25: Minimal Helper Functions for Training-Free TTA
# ============================================================
# WHY THIS EXISTS
# ───────────────
# Block 25-TTA-DIAG only needs two functions from the original
# Block 24: b24_augment (multi-view augmentation) and b24_run_dino
# (single inference pass). Running the full Block 24 here would
# be wasteful (calibration, gate, TTT loop we don't need) and
# subtly wrong (without Block 14, the optimizer setup inside
# Block 24 would silently target the FULL model's parameters,
# since nothing has frozen them in this fresh notebook).
#
# This is the full model frozen throughout — no LoRA, no
# trainable parameters, no weight updates anywhere in this
# notebook.
# ============================================================

import torch
import numpy as np
import random as _rnd
from PIL import Image as PILImage
import torchvision.transforms.functional as TF
import torchvision.transforms as T

print("=" * 65)
print("PRE-25: Minimal Helper Functions (augmentation + inference)")
print("=" * 65)
print()

# ─────────────────────────────────────────────────────────────
# Confirm model is fully frozen (expected in a fresh notebook
# with no LoRA injected — explicitly freezing here as a safety
# net regardless, since we never want gradients in this notebook)
# ─────────────────────────────────────────────────────────────
for p in dino_model.parameters():
    p.requires_grad = False
dino_model.eval()

n_trainable = sum(p.numel() for p in dino_model.parameters() if p.requires_grad)
print(f"✅ Model frozen — trainable params: {n_trainable}  (should be 0)")
print()

# ─────────────────────────────────────────────────────────────
# Multi-view augmentation (identical to Block 24's 24.2)
# ─────────────────────────────────────────────────────────────
def b24_augment(image_np: np.ndarray, seed: int) -> np.ndarray:
    _rnd.seed(seed * 31 + 7)
    img = PILImage.fromarray(image_np.astype(np.uint8))

    brightness = 0.85 + _rnd.random() * 0.30
    contrast   = 0.85 + _rnd.random() * 0.30
    img = TF.adjust_brightness(img, brightness)
    img = TF.adjust_contrast(img, contrast)

    if seed % 3 == 0:
        img = TF.adjust_saturation(img, 0.80 + _rnd.random() * 0.40)
    elif seed % 3 == 1:
        img = T.GaussianBlur(kernel_size=3, sigma=(0.1, 0.4))(img)
    else:
        img = TF.adjust_hue(img, (-0.05 + _rnd.random() * 0.10))

    return np.array(img)

# ─────────────────────────────────────────────────────────────
# Single inference pass (identical to Block 24's 24.3)
# ─────────────────────────────────────────────────────────────
def b24_run_dino(image_np: np.ndarray):
    dino_model.eval()
    with torch.no_grad():
        inputs = dino_processor(
            images=image_np,
            text=DINO_TEXT_PROMPT,
            return_tensors="pt"
        ).to(device)
        outputs = dino_model(**inputs)
        results = dino_processor.post_process_grounded_object_detection(
            outputs, inputs.input_ids,
            target_sizes=[image_np.shape[:2]],
            text_threshold=CRATTT_PARAMS["dino_text_thr"]
        )[0]

    label_key = "text_labels" if "text_labels" in results else "labels"
    raw_labels = results[label_key]
    labels = [str(l).lower().replace(".", "").strip() for l in raw_labels]

    return results["boxes"], results["scores"], labels

print("✅ b24_augment defined")
print("✅ b24_run_dino defined")
print()

# ─────────────────────────────────────────────────────────────
# Quick verification check on one image
# ─────────────────────────────────────────────────────────────
_test_img = loaded_images[image_files[0]]
_boxes, _scores, _labels = b24_run_dino(_test_img)
print(f"✅ Test inference OK — {len(_boxes)} detections on first clean image")

_aug_test = b24_augment(_test_img, seed=1)
print(f"✅ Augmentation OK — output shape {_aug_test.shape} (should match input)")
print()
print("=" * 65)
print("PRE-25 SETUP COMPLETE — ready for Block 25-TTA-DIAG")
print("=" * 65)

In [ ]:
# ============================================================
# BLOCK 25-TTA-DIAG: Score-SNR Confidence Recalibration
# Training-Free, Diagnostic at N=10
# Dada Victor Damilare | MRES7015 | University of Greater Manchester
# ============================================================
#
# WHY THIS BLOCK EXISTS
# ─────────────────────
# All TTT variants (confidence-only, regularised, full detection
# loss, rank=4 and rank=16) converged on the same negative/null
# result — the bottleneck was diagnosed as a gradient-clipping
# ceiling on adaptation magnitude, not supervision quality or
# capacity. This block tests a fundamentally different mechanism:
# NO weight updates at all. The multi-view consistency signal
# (already built and validated for reproducibility) is used to
# directly recalibrate confidence scores at inference time.
#
# CRITICAL DESIGN DIFFERENCE FROM THE TTT GATE:
# The internal TTRV gate's hard threshold (verify/discard) caused
# catastrophic failures when zero detections passed (n_verified=0),
# which we traced as the dominant driver of negative vs_baseline
# results throughout the TTT investigation. Recalibration has NO
# threshold — every detection is continuously reweighted, so this
# specific failure mode cannot occur here.
#
# CALIBRATING EXPECTATIONS: this is mechanistically simpler than
# even the weakest single component of FACTOR's ablation (their
# CSS-only variant: 21.61→25.55 mAP50, vs their full result of
# 29.30 which needs vision-language-grounded ASS/ACR we are not
# replicating). A modest improvement, if any, is the realistic
# bar here — not a FACTOR-scale result.
#
# This is a STARTING POINT. If it shows ANY positive signal, we
# scale up and refine from there.
# ============================================================

import torch
import numpy as np
import pandas as pd
import os
import random as _py_random

from imagecorruptions import corrupt as ic_corrupt
from torchvision.ops import box_iou
from tqdm.notebook import tqdm

print("=" * 65)
print("BLOCK 25-TTA-DIAG: Score-SNR Confidence Recalibration (N=10)")
print("=" * 65)
print()

# ─────────────────────────────────────────────────────────────
# Seeding helper (same pattern as all prior blocks)
# ─────────────────────────────────────────────────────────────
CORRUPTION_SEED_OFFSET = {
    "gaussian_noise": 1_000, "motion_blur": 2_000,
    "snow": 3_000, "contrast": 4_000,
}

def seed_for_corruption(img_id, corruption, severity):
    offset = CORRUPTION_SEED_OFFSET.get(corruption, 9_000)
    return (img_id * 100 + offset + severity) % (2**31)

def apply_corruption_deterministic(raw_img, img_id, corruption, severity):
    seed_val = seed_for_corruption(img_id, corruption, severity)
    np.random.seed(seed_val)
    _py_random.seed(seed_val)
    return ic_corrupt(raw_img, corruption_name=corruption, severity=severity)

# ─────────────────────────────────────────────────────────────
# Dependency check — note: no LoRA/tau dependencies needed here
# ─────────────────────────────────────────────────────────────
required = ["b24_augment", "b24_run_dino", "dino_model",
            "dino_processor", "DINO_TEXT_PROMPT", "device", "COCO_MAP"]
missing = [f for f in required if f not in globals()]
if missing:
    raise RuntimeError(f"Missing: {missing}\nRe-run Block 24 (helper functions only needed).")

dino_model.eval()  # frozen throughout — no training at all in this block
print("✅ Dependencies present. Model frozen (eval mode) — no weight updates in this block.")
print()

# ─────────────────────────────────────────────────────────────
# 25-TTA.1  MULTI-VIEW DETECTION WITH PER-DETECTION STATS
# (No filtering — every reference-view detection is kept and
# annotated with its multi-view consistency stats.)
# ─────────────────────────────────────────────────────────────
N_VIEWS  = 5
IOU_THR  = 0.40
SNR_EPS  = 1e-4

def multiview_detections(image_np):
    ref_boxes, ref_scores, ref_labels = b24_run_dino(image_np)
    if len(ref_boxes) == 0:
        return []

    view_outputs = []
    for v in range(1, N_VIEWS):
        aug = b24_augment(image_np, seed=v)
        vb, vs, vl = b24_run_dino(aug)
        view_outputs.append((vb, vs, vl))

    detections = []
    for ref_box, ref_score, ref_label in zip(ref_boxes, ref_scores, ref_labels):
        if ref_label not in COCO_MAP:
            continue

        scores_seen = [ref_score.item()]
        match_count = 1
        rb_exp = ref_box.unsqueeze(0)

        for (vb, vs, _) in view_outputs:
            if len(vb) == 0:
                continue
            ious = box_iou(rb_exp, vb)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= IOU_THR:
                scores_seen.append(vs[best_j].item())
                match_count += 1

        arr        = np.array(scores_seen)
        score_mean = float(arr.mean())
        score_std  = float(arr.std())
        s_snr      = score_mean / (score_std + SNR_EPS)
        iou_cons   = match_count / N_VIEWS

        detections.append({
            "box": ref_box, "label": ref_label,
            "score_orig": ref_score.item(),
            "score_mean": score_mean,
            "s_snr": s_snr, "iou_consensus": iou_cons,
        })
    return detections

print("✅ Multi-view detection function defined (no gating/filtering)")
print()

# ─────────────────────────────────────────────────────────────
# 25-TTA.2  Configuration
# ─────────────────────────────────────────────────────────────
DIAG_CORRUPTIONS = ["motion_blur", "contrast"]
DIAG_SEVERITY    = 5
DIAG_N           = 10

print(f"Corruptions : {DIAG_CORRUPTIONS}")
print(f"Severity    : {DIAG_SEVERITY}")
print(f"N images    : {DIAG_N}")
print()

# ─────────────────────────────────────────────────────────────
# 25-TTA.3  Main loop — Baseline vs Variant A vs Variant B
# ─────────────────────────────────────────────────────────────
results = []

for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[:DIAG_N], desc=f"{corruption} sev{DIAG_SEVERITY}"):

        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(
            raw_img, img_id, corruption, DIAG_SEVERITY
        )

        # ── Baseline: raw single-pass frozen detection ──────────
        boxes_b, scores_b, labels_b = b24_run_dino(c_img)
        preds_base = [
            {"image_id": img_id, "category_id": COCO_MAP[l],
             "bbox": [b[0].item(), b[1].item(),
                      (b[2]-b[0]).item(), (b[3]-b[1]).item()],
             "score": s.item()}
            for b, s, l in zip(boxes_b, scores_b, labels_b)
            if l in COCO_MAP
        ]
        mAP_base, _ = compute_map(preds_base, coco_gt, [img_id])

        # ── Multi-view detections (shared by both variants) ─────
        dets = multiview_detections(c_img)

        if len(dets) == 0:
            for variant in ["mean_score", "consensus_weighted"]:
                results.append({
                    "corruption": corruption, "img_id": img_id, "variant": variant,
                    "mAP_baseline": mAP_base, "mAP_recal": mAP_base,
                    "vs_baseline": 0.0, "beats_baseline": False,
                    "n_detections": 0,
                })
            continue

        # ── Variant A: mean score across views ──────────────────
        preds_a = [
            {"image_id": img_id, "category_id": COCO_MAP[d["label"]],
             "bbox": [d["box"][0].item(), d["box"][1].item(),
                      (d["box"][2]-d["box"][0]).item(), (d["box"][3]-d["box"][1]).item()],
             "score": d["score_mean"]}
            for d in dets
        ]
        mAP_a, _ = compute_map(preds_a, coco_gt, [img_id])
        results.append({
            "corruption": corruption, "img_id": img_id, "variant": "mean_score",
            "mAP_baseline": mAP_base, "mAP_recal": mAP_a,
            "vs_baseline": mAP_a - mAP_base, "beats_baseline": mAP_a > mAP_base,
            "n_detections": len(dets),
        })

        # ── Variant B: mean score × consensus weight ─────────────
        preds_b = [
            {"image_id": img_id, "category_id": COCO_MAP[d["label"]],
             "bbox": [d["box"][0].item(), d["box"][1].item(),
                      (d["box"][2]-d["box"][0]).item(), (d["box"][3]-d["box"][1]).item()],
             "score": d["score_mean"] * d["iou_consensus"]}
            for d in dets
        ]
        mAP_b, _ = compute_map(preds_b, coco_gt, [img_id])
        results.append({
            "corruption": corruption, "img_id": img_id, "variant": "consensus_weighted",
            "mAP_baseline": mAP_base, "mAP_recal": mAP_b,
            "vs_baseline": mAP_b - mAP_base, "beats_baseline": mAP_b > mAP_base,
            "n_detections": len(dets),
        })

# ─────────────────────────────────────────────────────────────
# 25-TTA.4  Results
# ─────────────────────────────────────────────────────────────
df_tta = pd.DataFrame(results)

print()
print("=" * 65)
print("RESULTS — Training-Free Recalibration vs Frozen Baseline")
print("=" * 65)

summary = (
    df_tta.groupby(["corruption", "variant"])
    .agg(vs_Baseline=("vs_baseline", "mean"),
         Beats=("beats_baseline", "sum"),
         N=("beats_baseline", "count"),
         AvgDetections=("n_detections", "mean"))
    .round(4)
)
print(summary.to_string())

print()
overall = (
    df_tta.groupby("variant")
    .agg(vs_Baseline=("vs_baseline", "mean"),
         Beats=("beats_baseline", "sum"),
         N=("beats_baseline", "count"))
    .round(4)
)
print(overall.to_string())

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_tta.to_csv("/kaggle/working/tables/table_25tta_diag_n10.csv", index=False)
print("\n✅ Saved: table_25tta_diag_n10.csv")
print()
print("=" * 65)
print("BLOCK 25-TTA-DIAG COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# BLOCK 25-TTA-DIAG.5: Correlation Check
# Does the consistency signal carry independent information,
# or is it just redundant with the original score?
# ============================================================
#
# WHY THIS CHECK MATTERS
# ───────────────────────
# mAP is rank-based. If iou_consensus/s_snr are highly correlated
# with the original score (Spearman ≈ 1.0), then any recalibration
# built from them will be nearly rank-preserving — and rank-
# preserving transformations cannot change AP, no matter how the
# absolute scores shift. This directly tests whether there's even
# enough independent information in the signal to ever reorder
# detections, before we design anything more sophisticated.
# ============================================================

from scipy.stats import spearmanr, pearsonr
import pandas as pd
import numpy as np

print("=" * 65)
print("Correlation Check: original score vs. consistency signals")
print("=" * 65)
print()

det_records = []

for corruption in DIAG_CORRUPTIONS:
    for img_path in image_files[:DIAG_N]:
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        dets = multiview_detections(c_img)
        for d in dets:
            det_records.append({
                "corruption": corruption, "img_id": img_id,
                "score_orig": d["score_orig"],
                "score_mean": d["score_mean"],
                "s_snr": d["s_snr"],
                "iou_consensus": d["iou_consensus"],
            })

df_corr = pd.DataFrame(det_records)
print(f"Total detections pooled: {len(df_corr)}")
print()

# ─────────────────────────────────────────────────────────────
# Overall correlations
# ─────────────────────────────────────────────────────────────
for sig in ["iou_consensus", "s_snr", "score_mean"]:
    sp, sp_p = spearmanr(df_corr["score_orig"], df_corr[sig])
    pe, pe_p = pearsonr(df_corr["score_orig"], df_corr[sig])
    print(f"score_orig vs {sig:14s}  Spearman: {sp:+.3f} (p={sp_p:.4f})  "
          f"Pearson: {pe:+.3f} (p={pe_p:.4f})")

print()
print("─" * 65)
print("Per-corruption breakdown")
print("─" * 65)
for corruption in DIAG_CORRUPTIONS:
    df_c = df_corr[df_corr["corruption"] == corruption]
    print(f"\n{corruption} (N={len(df_c)} detections):")
    for sig in ["iou_consensus", "s_snr"]:
        sp, sp_p = spearmanr(df_c["score_orig"], df_c[sig])
        print(f"  score_orig vs {sig:14s}  Spearman: {sp:+.3f} (p={sp_p:.4f})")

# ─────────────────────────────────────────────────────────────
# Disagreement cases: score high but consensus low, or vice versa
# (within each image, since that's the relevant scope for AP
# ranking — detections compete against each other within a
# category across the dataset, but let's first check within-image
# disagreement as the simplest signal of reordering potential)
# ─────────────────────────────────────────────────────────────
print()
print("─" * 65)
print("Disagreement check: score in top tercile, consensus in bottom (or vice versa)")
print("─" * 65)

disagree_count = 0
total_checked  = 0

for (corruption, img_id), group in df_corr.groupby(["corruption", "img_id"]):
    if len(group) < 3:
        continue  # need at least 3 detections for terciles to mean anything
    score_rank = group["score_orig"].rank(pct=True)
    cons_rank  = group["iou_consensus"].rank(pct=True)

    high_score_low_cons = ((score_rank > 0.66) & (cons_rank < 0.33)).sum()
    low_score_high_cons = ((score_rank < 0.33) & (cons_rank > 0.66)).sum()

    disagree_count += high_score_low_cons + low_score_high_cons
    total_checked  += len(group)

print(f"Detections in a tercile-disagreement position: {disagree_count} / {total_checked}  "
      f"({100*disagree_count/total_checked:.1f}%)")
print()
print("=" * 65)
print("CORRELATION CHECK COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# BLOCK 25-TTA-DIAG.6: Correlation Check with STRONGER Perturbations
# ============================================================
#
# WHY THIS VERSION EXISTS
# ─────────────────────
# The mild jitter in b24_augment (±15% brightness/contrast) showed
# near-zero disagreement between score and consistency signals —
# likely too gentle to reveal which detections are actually
# fragile. This version uses 6 distinct, STRONG, single-attribute
# perturbations (one per view) instead of several mild randomized
# factors combined — stronger magnitude, and individually
# interpretable (we can see which attribute type drives
# instability, if any does).
#
# Each view is a FIXED, deterministic transform — no randomness
# needed here at all, since the perturbation itself doesn't vary
# run to run. This is even more reproducible than the original
# design.
# ============================================================

import torch
import numpy as np
from PIL import Image as PILImage
import torchvision.transforms.functional as TF
import torchvision.transforms as T
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from torchvision.ops import box_iou
from tqdm.notebook import tqdm
import os

print("=" * 65)
print("BLOCK 25-TTA-DIAG.6: Stronger Perturbations — Correlation Check")
print("=" * 65)
print()

# ─────────────────────────────────────────────────────────────
# Six distinct, strong, single-attribute perturbations
# ─────────────────────────────────────────────────────────────
def strong_view(image_np: np.ndarray, view_idx: int) -> np.ndarray:
    img = PILImage.fromarray(image_np.astype(np.uint8))

    if view_idx == 1:      # Strong brightness reduction
        img = TF.adjust_brightness(img, 0.45)
    elif view_idx == 2:    # Strong contrast reduction
        img = TF.adjust_contrast(img, 0.45)
    elif view_idx == 3:    # Strong blur
        img = T.GaussianBlur(kernel_size=9, sigma=2.5)(img)
    elif view_idx == 4:    # Strong additive noise
        arr = np.array(img).astype(np.float32)
        noise = np.random.RandomState(42).normal(0, 35, arr.shape)  # fixed seed → deterministic
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img = PILImage.fromarray(arr)
    elif view_idx == 5:    # Strong saturation reduction
        img = TF.adjust_saturation(img, 0.25)
    elif view_idx == 6:    # Strong hue shift
        img = TF.adjust_hue(img, 0.15)

    return np.array(img)

VIEW_NAMES = {1: "brightness", 2: "contrast", 3: "blur",
              4: "noise", 5: "saturation", 6: "hue"}
N_STRONG_VIEWS = 6
IOU_THR = 0.40
SNR_EPS = 1e-4

print(f"✅ 6 strong single-attribute views defined: {list(VIEW_NAMES.values())}")
print()

def multiview_detections_strong(image_np):
    ref_boxes, ref_scores, ref_labels = b24_run_dino(image_np)
    if len(ref_boxes) == 0:
        return []

    view_outputs = []
    for v in range(1, N_STRONG_VIEWS + 1):
        aug = strong_view(image_np, v)
        vb, vs, vl = b24_run_dino(aug)
        view_outputs.append((v, vb, vs, vl))

    detections = []
    for ref_box, ref_score, ref_label in zip(ref_boxes, ref_scores, ref_labels):
        if ref_label not in COCO_MAP:
            continue

        scores_seen   = [ref_score.item()]
        match_count   = 1
        matched_views = []
        rb_exp = ref_box.unsqueeze(0)

        for (v_idx, vb, vs, _) in view_outputs:
            if len(vb) == 0:
                continue
            ious = box_iou(rb_exp, vb)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= IOU_THR:
                scores_seen.append(vs[best_j].item())
                match_count += 1
                matched_views.append(v_idx)

        arr        = np.array(scores_seen)
        score_mean = float(arr.mean())
        score_std  = float(arr.std())
        s_snr      = score_mean / (score_std + SNR_EPS)
        iou_cons   = match_count / (N_STRONG_VIEWS + 1)

        detections.append({
            "box": ref_box, "label": ref_label,
            "score_orig": ref_score.item(),
            "score_mean": score_mean,
            "s_snr": s_snr, "iou_consensus": iou_cons,
            "matched_views": [VIEW_NAMES[v] for v in matched_views],
        })
    return detections

print("✅ multiview_detections_strong defined")
print()

# ─────────────────────────────────────────────────────────────
# Run on the same N=10, 2 corruptions, severity 5
# ─────────────────────────────────────────────────────────────
det_records = []

for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[:DIAG_N], desc=f"{corruption} sev{DIAG_SEVERITY}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        dets = multiview_detections_strong(c_img)
        for d in dets:
            det_records.append({
                "corruption": corruption, "img_id": img_id,
                "score_orig": d["score_orig"], "score_mean": d["score_mean"],
                "s_snr": d["s_snr"], "iou_consensus": d["iou_consensus"],
                "n_matched_views": len(d["matched_views"]),
            })

df_strong = pd.DataFrame(det_records)
print(f"\nTotal detections pooled: {len(df_strong)}")
print()

# ─────────────────────────────────────────────────────────────
# Correlation check — same structure as before, for direct comparison
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("CORRELATION RESULTS — Strong Perturbations")
print("=" * 65)
print()

for sig in ["iou_consensus", "s_snr", "score_mean"]:
    sp, sp_p = spearmanr(df_strong["score_orig"], df_strong[sig])
    pe, pe_p = pearsonr(df_strong["score_orig"], df_strong[sig])
    print(f"score_orig vs {sig:14s}  Spearman: {sp:+.3f} (p={sp_p:.4f})  "
          f"Pearson: {pe:+.3f} (p={pe_p:.4f})")

print()
print("─" * 65)
print("Per-corruption breakdown")
print("─" * 65)
for corruption in DIAG_CORRUPTIONS:
    df_c = df_strong[df_strong["corruption"] == corruption]
    print(f"\n{corruption} (N={len(df_c)} detections):")
    for sig in ["iou_consensus", "s_snr"]:
        sp, sp_p = spearmanr(df_c["score_orig"], df_c[sig])
        print(f"  score_orig vs {sig:14s}  Spearman: {sp:+.3f} (p={sp_p:.4f})")

# ─────────────────────────────────────────────────────────────
# Disagreement check (same tercile method, same scope, for
# direct comparison against the mild-perturbation result)
# ─────────────────────────────────────────────────────────────
print()
print("─" * 65)
print("Disagreement check: score top tercile / consensus bottom tercile (or vice versa)")
print("─" * 65)

disagree_count = 0
total_checked  = 0

for (corruption, img_id), group in df_strong.groupby(["corruption", "img_id"]):
    if len(group) < 3:
        continue
    score_rank = group["score_orig"].rank(pct=True)
    cons_rank  = group["iou_consensus"].rank(pct=True)

    high_score_low_cons = ((score_rank > 0.66) & (cons_rank < 0.33)).sum()
    low_score_high_cons = ((score_rank < 0.33) & (cons_rank > 0.66)).sum()

    disagree_count += high_score_low_cons + low_score_high_cons
    total_checked  += len(group)

print(f"Detections in a tercile-disagreement position: {disagree_count} / {total_checked}  "
      f"({100*disagree_count/total_checked:.1f}%)"
      if total_checked > 0 else "Not enough multi-detection images to check.")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_strong.to_csv("/kaggle/working/tables/table_25tta_strong_correlation.csv", index=False)
print(f"\n✅ Saved: table_25tta_strong_correlation.csv")
print()
print("=" * 65)
print("DIAG.6 COMPLETE")
print("=" * 65)

In [9]:
# ============================================================
# Category-Distribution Extraction Utility — Step 1
# Dada Victor Damilare | MRES7015 | University of Greater Manchester
# ============================================================
#
# WHY THIS EXISTS
# ─────────────────────
# Everything built so far only uses the SINGLE best-matching
# category's score per detection. KL divergence needs the FULL
# probability distribution across all categories. GroundingDINO
# doesn't expose this directly — it outputs token-level alignment
# logits, and category scores come from matching token spans in
# the text prompt to category names. This utility builds that
# mapping and validates it carefully, since a silent indexing
# mistake here would produce wrong distributions with no obvious
# error downstream.
# ============================================================

import torch
import os
import random as _r

print("=" * 65)
print("STEP 1: Category-Distribution Extraction Utility")
print("=" * 65)
print()

# ─────────────────────────────────────────────────────────────
# 1.1  Category order — must match the label strings used
# elsewhere in the pipeline (COCO_MAP keys)
# ─────────────────────────────────────────────────────────────
category_order = list(COCO_MAP.keys())
print(f"Category order: {len(category_order)} categories")
print(f"First 5: {category_order[:5]}")
print(f"Last 5 : {category_order[-5:]}")
print()

# ─────────────────────────────────────────────────────────────
# 1.2  Tokenize the prompt WITH character offsets
# ─────────────────────────────────────────────────────────────
tokenizer = dino_processor.tokenizer
tok_with_offsets = tokenizer(
    DINO_TEXT_PROMPT,
    return_offsets_mapping=True,
    add_special_tokens=True,
    return_tensors=None,
)
offsets        = tok_with_offsets["offset_mapping"]
token_ids_ref  = tok_with_offsets["input_ids"]

print(f"Tokenized prompt length: {len(token_ids_ref)} tokens")
print()

# ─────────────────────────────────────────────────────────────
# 1.3  CRITICAL CHECK: does this tokenization match what the
# model actually receives? If not, every span mapping below
# would be silently wrong.
# ─────────────────────────────────────────────────────────────
_sanity_inputs = dino_processor(
    images=loaded_images[image_files[0]],
    text=DINO_TEXT_PROMPT,
    return_tensors="pt"
).to(device)
actual_input_ids = _sanity_inputs.input_ids[0].cpu().tolist()

if token_ids_ref != actual_input_ids:
    print("❌ MISMATCH between offset-mapping tokenization and actual model inputs!")
    print(f"   Offset-based length : {len(token_ids_ref)}")
    print(f"   Model input length  : {len(actual_input_ids)}")
    raise RuntimeError(
        "Tokenization mismatch — category span mapping would be WRONG. "
        "Stop and investigate before proceeding."
    )
else:
    print(f"✅ Tokenization confirmed IDENTICAL to model inputs ({len(token_ids_ref)} tokens)")
print()

# ─────────────────────────────────────────────────────────────
# 1.4  Build category → token-index spans via character offsets
# ─────────────────────────────────────────────────────────────
def build_category_token_spans(text_prompt, category_names, offsets):
    spans = {}
    not_found = []
    search_start = 0
    for cat in category_names:
        idx = text_prompt.find(cat, search_start)
        if idx == -1:
            idx = text_prompt.lower().find(cat.lower(), search_start)
        if idx == -1:
            spans[cat] = []
            not_found.append(cat)
            continue
        start_char, end_char = idx, idx + len(cat)
        search_start = end_char

        token_ids = [
            i for i, (s, e) in enumerate(offsets)
            if not (s == 0 and e == 0) and s < end_char and e > start_char
        ]
        spans[cat] = token_ids
    return spans, not_found

category_token_spans, not_found = build_category_token_spans(
    DINO_TEXT_PROMPT, category_order, offsets
)

print(f"✅ Built token spans for {len(category_order) - len(not_found)}/{len(category_order)} categories")
if not_found:
    print(f"⚠️  NOT FOUND in prompt (will get zero score, investigate if this list is non-empty): {not_found}")
print()

# ─────────────────────────────────────────────────────────────
# 1.5  Sanity check: decode tokens back to text for random samples
# ─────────────────────────────────────────────────────────────
print("─" * 65)
print("Sanity check: decoded tokens for 5 random categories")
print("─" * 65)
_r.seed(0)
valid_cats = [c for c in category_order if category_token_spans[c]]
sample_cats = _r.sample(valid_cats, min(5, len(valid_cats)))
for cat in sample_cats:
    tids = category_token_spans[cat]
    decoded = tokenizer.decode([token_ids_ref[i] for i in tids])
    norm_decoded = decoded.strip().lower().replace(" ", "")
    norm_cat     = cat.lower().replace(" ", "")
    match_flag = "✅" if (norm_decoded in norm_cat or norm_cat in norm_decoded) else "⚠️ CHECK THIS"
    print(f"  '{cat}' → tokens {tids} → decoded: '{decoded}'  {match_flag}")
print()

# ─────────────────────────────────────────────────────────────
# 1.6  Category distribution extraction function
# Returns BOTH raw sigmoid scores (for thresholding) and
# softmax-normalised distribution (for KL divergence later)
# ─────────────────────────────────────────────────────────────
def get_category_distribution(outputs, category_token_spans, category_order):
    logits = outputs.logits[0]          # [num_queries, num_tokens]
    scores_sigmoid = logits.sigmoid()   # [num_queries, num_tokens]

    cat_score_cols = []
    for cat in category_order:
        tids = category_token_spans.get(cat, [])
        if len(tids) == 0:
            cat_score_cols.append(torch.zeros(scores_sigmoid.shape[0], device=scores_sigmoid.device))
        else:
            cat_score_cols.append(scores_sigmoid[:, tids].max(dim=-1).values)

    cat_scores_raw = torch.stack(cat_score_cols, dim=-1)    # [num_queries, num_categories] — for thresholding
    cat_dist       = torch.softmax(cat_scores_raw, dim=-1)  # [num_queries, num_categories] — for KL divergence
    return cat_scores_raw, cat_dist

print("✅ get_category_distribution defined")
print()

# ─────────────────────────────────────────────────────────────
# 1.7  Full pipeline function: run inference, threshold using
# RAW scores (matching existing convention), keep query indices
# and full distributions for later KL-divergence use
# ─────────────────────────────────────────────────────────────
def run_dino_with_categories(image_np, threshold=None):
    if threshold is None:
        threshold = CRATTT_PARAMS["dino_text_thr"]

    dino_model.eval()
    with torch.no_grad():
        inputs = dino_processor(images=image_np, text=DINO_TEXT_PROMPT, return_tensors="pt").to(device)
        outputs = dino_model(**inputs)

    cat_scores_raw, cat_dist = get_category_distribution(outputs, category_token_spans, category_order)
    max_raw_scores, max_cat_idx = cat_scores_raw.max(dim=-1)  # thresholding uses RAW scores

    keep_mask    = max_raw_scores >= threshold
    keep_indices = keep_mask.nonzero(as_tuple=True)[0]

    if len(keep_indices) == 0:
        return [], [], [], [], None

    img_h, img_w = image_np.shape[:2]
    pred_cxcywh = outputs.pred_boxes[0]
    cx = pred_cxcywh[:, 0] * img_w
    cy = pred_cxcywh[:, 1] * img_h
    pw = pred_cxcywh[:, 2] * img_w
    ph = pred_cxcywh[:, 3] * img_h
    pred_xyxy = torch.stack([cx - pw/2, cy - ph/2, cx + pw/2, cy + ph/2], dim=-1)

    boxes     = pred_xyxy[keep_indices]
    scores    = max_raw_scores[keep_indices]
    labels    = [category_order[i] for i in max_cat_idx[keep_indices].tolist()]
    query_idx = keep_indices.tolist()
    full_dist = cat_dist[keep_indices]   # [num_kept, num_categories] — needed for KL divergence next step

    return boxes, scores, labels, query_idx, full_dist

print("✅ run_dino_with_categories defined")
print()

# ─────────────────────────────────────────────────────────────
# 1.8  VALIDATION: compare against the existing, already-trusted
# b24_run_dino on real images — checking label-set agreement,
# not exact box-by-box matching (different pooling internals are
# expected to cause minor differences; large disagreement would
# signal a real bug)
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("VALIDATION: new extraction vs existing b24_run_dino")
print("=" * 65)

for img_path in image_files[:3]:
    img = loaded_images[img_path]

    boxes_old, scores_old, labels_old = b24_run_dino(img)
    boxes_new, scores_new, labels_new, qidx_new, dist_new = run_dino_with_categories(img)

    set_old = sorted(set(labels_old))
    set_new = sorted(set(labels_new))
    overlap = set(set_old) & set(set_new)

    print(f"\n{os.path.basename(img_path)}:")
    print(f"  Existing (library post-process): {len(labels_old)} dets — labels: {set_old}")
    print(f"  New (custom extraction)        : {len(labels_new)} dets — labels: {set_new}")
    print(f"  Label-set overlap: {len(overlap)}/{len(set(set_old) | set(set_new))} "
          f"{'✅ good agreement' if len(overlap) >= max(1, len(set_old)-1) else '⚠️  check this one'}")

print()
print("=" * 65)
print("STEP 1 COMPLETE")
print("=" * 65)

STEP 1: Category-Distribution Extraction Utility

Category order: 80 categories
First 5: ['person', 'bicycle', 'car', 'motorcycle', 'airplane']
Last 5 : ['vase', 'scissors', 'teddy bear', 'hair drier', 'toothbrush']

Tokenized prompt length: 195 tokens

✅ Tokenization confirmed IDENTICAL to model inputs (195 tokens)

✅ Built token spans for 80/80 categories

─────────────────────────────────────────────────────────────────
Sanity check: decoded tokens for 5 random categories
─────────────────────────────────────────────────────────────────
  'orange' → tokens [119] → decoded: 'orange'  ✅
  'pizza' → tokens [130] → decoded: 'pizza'  ✅
  'bus' → tokens [11] → decoded: 'bus'  ✅
  'kite' → tokens [80] → decoded: 'kite'  ✅
  'remote' → tokens [158] → decoded: 'remote'  ✅

✅ get_category_distribution defined

✅ run_dino_with_categories defined

VALIDATION: new extraction vs existing b24_run_dino


NameError: name 'b24_run_dino' is not defined

In [ ]:
import inspect
print(inspect.signature(dino_processor.post_process_grounded_object_detection))

In [ ]:
print(CRATTT_PARAMS["dino_text_thr"])

In [ ]:
# ─────────────────────────────────────────────────────────────
# CORRECTED run_dino_with_categories — uses the library's actual
# box-confidence gate (0.25), not the text-match threshold (0.12)
# ─────────────────────────────────────────────────────────────
BOX_THRESHOLD = 0.25  # library default 'threshold' param —
                       # implicitly active throughout Blocks 24-24e,
                       # now made explicit for this utility

def run_dino_with_categories(image_np, box_threshold=None):
    if box_threshold is None:
        box_threshold = BOX_THRESHOLD

    dino_model.eval()
    with torch.no_grad():
        inputs = dino_processor(images=image_np, text=DINO_TEXT_PROMPT, return_tensors="pt").to(device)
        outputs = dino_model(**inputs)

    cat_scores_raw, cat_dist = get_category_distribution(outputs, category_token_spans, category_order)
    max_raw_scores, max_cat_idx = cat_scores_raw.max(dim=-1)  # box-confidence-equivalent score

    keep_mask    = max_raw_scores >= box_threshold   # ← FIXED: box gate (0.25), not text gate (0.12)
    keep_indices = keep_mask.nonzero(as_tuple=True)[0]

    if len(keep_indices) == 0:
        return [], [], [], [], None

    img_h, img_w = image_np.shape[:2]
    pred_cxcywh = outputs.pred_boxes[0]
    cx = pred_cxcywh[:, 0] * img_w
    cy = pred_cxcywh[:, 1] * img_h
    pw = pred_cxcywh[:, 2] * img_w
    ph = pred_cxcywh[:, 3] * img_h
    pred_xyxy = torch.stack([cx - pw/2, cy - ph/2, cx + pw/2, cy + ph/2], dim=-1)

    boxes     = pred_xyxy[keep_indices]
    scores    = max_raw_scores[keep_indices]
    labels    = [category_order[i] for i in max_cat_idx[keep_indices].tolist()]
    query_idx = keep_indices.tolist()
    full_dist = cat_dist[keep_indices]

    return boxes, scores, labels, query_idx, full_dist

print(f"✅ run_dino_with_categories corrected — using box_threshold={BOX_THRESHOLD}")
print()

# ─────────────────────────────────────────────────────────────
# Re-run the SAME validation to check whether counts now align
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("RE-VALIDATION: corrected extraction vs existing b24_run_dino")
print("=" * 65)

for img_path in image_files[:3]:
    img = loaded_images[img_path]

    boxes_old, scores_old, labels_old = b24_run_dino(img)
    boxes_new, scores_new, labels_new, qidx_new, dist_new = run_dino_with_categories(img)

    set_old = sorted(set(labels_old))
    set_new = sorted(set(labels_new))
    overlap = set(set_old) & set(set_new)

    print(f"\n{os.path.basename(img_path)}:")
    print(f"  Existing (library post-process): {len(labels_old)} dets — labels: {set_old}")
    print(f"  New (custom extraction)        : {len(labels_new)} dets — labels: {set_new}")
    print(f"  Label-set overlap: {len(overlap)}/{len(set(set_old) | set(set_new))} "
          f"{'✅ good agreement' if len(overlap) >= max(1, len(set_old)-1) else '⚠️  check this one'}")

In [ ]:
# ============================================================
# Per-query verification: does my argmax label match one of
# the words in a composite phrase, for the SAME underlying query?
# ============================================================
from torchvision.ops import box_iou

print("=" * 65)
print("Per-Query Verification: Composite Phrase vs Clean Argmax")
print("=" * 65)
print()

test_img_path = image_files[0]  # 000000000139.jpg
test_img = loaded_images[test_img_path]

# ── Library's output (composite phrases) ──
boxes_old, scores_old, labels_old = b24_run_dino(test_img)

# ── My output (clean argmax + full distribution) ──
boxes_new, scores_new, labels_new, qidx_new, dist_new = run_dino_with_categories(test_img)

# Find a composite-labeled detection to investigate
target_label = None
target_idx = None
for i, lbl in enumerate(labels_old):
    if lbl not in category_order and len(lbl.split()) > 2:  # likely a multi-category composite
        target_label = lbl
        target_idx = i
        break

if target_label is None:
    print("⚠️  No clear composite label found in this image — try a different image.")
else:
    print(f"Investigating library detection: '{target_label}'  (score={scores_old[target_idx]:.3f})")
    target_box = boxes_old[target_idx].unsqueeze(0)

    # Find the best-matching box in MY output via IoU
    ious = box_iou(target_box, boxes_new)
    best_iou, best_j = ious[0].max(0)
    best_j = best_j.item()

    print(f"Best-matching box in my extraction: IoU={best_iou.item():.4f}  "
          f"(should be very close to 1.0 if it's the same query)")
    print(f"My argmax label for this query: '{labels_new[best_j]}'")
    print(f"Score: {scores_new[best_j]:.3f}")
    print()

    # Decompose the composite label into individual category names it might contain
    contained_cats = [c for c in category_order if c in target_label.split()
                       or all(w in target_label for w in c.split())]
    print(f"Individual categories detectable within the composite phrase: {contained_cats}")
    is_consistent = labels_new[best_j] in contained_cats
    print(f"My label is one of these: {'✅ YES — consistent' if is_consistent else '⚠️ NO — investigate'}")
    print()

    # Show the full probability distribution's top categories for this query
    print("─" * 65)
    print(f"Full category distribution for this query (top 8 by probability)")
    print("─" * 65)
    dist_for_query = dist_new[best_j]
    top_probs, top_idx = dist_for_query.topk(8)
    for p, idx in zip(top_probs.tolist(), top_idx.tolist()):
        cat = category_order[idx]
        flag = " ← contained in composite phrase" if cat in contained_cats else ""
        print(f"  {cat:20s}  {p:.4f}{flag}")

print()
print("=" * 65)
print("VERIFICATION COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# STEP 2: KL-Divergence Correlation Check
# ============================================================
import torch
import numpy as np
import pandas as pd
from torchvision.ops import box_iou
from scipy.stats import spearmanr, pearsonr
from tqdm.notebook import tqdm
import os

print("=" * 65)
print("STEP 2: KL-Divergence Across Views — Correlation Check")
print("=" * 65)
print()

IOU_THR_KL = 0.40

def kl_divergence(p_ref, p_view, eps=1e-8):
    # Forward KL(P_ref || P_view) — weighted toward categories where
    # the reference was already confident, per FACTOR's formulation
    return torch.sum(p_ref * (torch.log(p_ref + eps) - torch.log(p_view + eps))).item()

def kl_records_for_image(c_img):
    boxes_ref, scores_ref, labels_ref, qidx_ref, dist_ref = run_dino_with_categories(c_img)
    if len(boxes_ref) == 0:
        return []

    view_data = []
    for v in range(1, N_STRONG_VIEWS + 1):
        aug = strong_view(c_img, v)
        boxes_v, scores_v, labels_v, qidx_v, dist_v = run_dino_with_categories(aug)
        view_data.append((boxes_v, dist_v))

    records = []
    for i in range(len(boxes_ref)):
        ref_box = boxes_ref[i].unsqueeze(0)
        kls = []
        for (boxes_v, dist_v) in view_data:
            if len(boxes_v) == 0:
                continue
            ious = box_iou(ref_box, boxes_v)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= IOU_THR_KL:
                kl = kl_divergence(dist_ref[i], dist_v[best_j.item()])
                kls.append(kl)

        records.append({
            "score_orig": scores_ref[i].item(),
            "label": labels_ref[i],
            "kl_avg": float(np.mean(kls)) if kls else np.nan,
            "n_matched_views": len(kls),
        })
    return records

print("✅ KL-divergence computation function defined")
print()

# ─────────────────────────────────────────────────────────────
# Run on the same N=10, 2 corruptions, severity 5
# ─────────────────────────────────────────────────────────────
det_records_kl = []

for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[:DIAG_N], desc=f"{corruption} sev{DIAG_SEVERITY}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        recs = kl_records_for_image(c_img)
        for r in recs:
            r["corruption"] = corruption
            r["img_id"] = img_id
            det_records_kl.append(r)

df_kl = pd.DataFrame(det_records_kl)
print(f"\nTotal detections: {len(df_kl)}")
print(f"Detections with at least 1 matched view: {(df_kl['n_matched_views'] > 0).sum()}")
print()

df_valid = df_kl.dropna(subset=["kl_avg"]).copy()

# ─────────────────────────────────────────────────────────────
# Correlation check
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("CORRELATION RESULTS — KL Divergence vs Original Score")
print("=" * 65)
sp, sp_p = spearmanr(df_valid["score_orig"], df_valid["kl_avg"])
pe, pe_p = pearsonr(df_valid["score_orig"], df_valid["kl_avg"])
print(f"score_orig vs kl_avg   Spearman: {sp:+.3f} (p={sp_p:.4f})  Pearson: {pe:+.3f} (p={pe_p:.4f})")
print(f"(Expected direction is NEGATIVE — confident detections should diverge less under perturbation)")
print()

for corruption in DIAG_CORRUPTIONS:
    df_c = df_valid[df_valid["corruption"] == corruption]
    sp, sp_p = spearmanr(df_c["score_orig"], df_c["kl_avg"])
    print(f"{corruption}: Spearman {sp:+.3f} (p={sp_p:.4f}), N={len(df_c)}")

# ─────────────────────────────────────────────────────────────
# Disagreement check — cases that VIOLATE the expected negative
# relationship (high score + high KL, or low score + low KL)
# ─────────────────────────────────────────────────────────────
print()
print("─" * 65)
print("Disagreement check: violations of expected negative score↔KL relationship")
print("─" * 65)

disagree_count = 0
total_checked  = 0

for (corruption, img_id), group in df_valid.groupby(["corruption", "img_id"]):
    if len(group) < 3:
        continue
    score_rank = group["score_orig"].rank(pct=True)
    kl_rank    = group["kl_avg"].rank(pct=True)

    confident_but_unstable = ((score_rank > 0.66) & (kl_rank > 0.66)).sum()
    unconfident_but_stable = ((score_rank < 0.33) & (kl_rank < 0.33)).sum()

    disagree_count += confident_but_unstable + unconfident_but_stable
    total_checked  += len(group)

print(f"Detections violating the expected pattern: {disagree_count} / {total_checked}  "
      f"({100*disagree_count/total_checked:.1f}%)" if total_checked > 0 else "Not enough data.")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_kl.to_csv("/kaggle/working/tables/table_step2_kl_correlation.csv", index=False)
print(f"\n✅ Saved: table_step2_kl_correlation.csv")
print()
print("=" * 65)
print("STEP 2 COMPLETE")
print("=" * 65)

In [ ]:
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr
import os

print("=" * 65)
print("Per-Image Relative Gating: FACTOR's Actual CSS Formula")
print("=" * 65)
print()
print("Reusing df_valid from Step 2 — no new forward passes needed.")
print()

df_css = df_valid.copy()

# Only keep images with enough detections for a meaningful per-image average
group_sizes = df_css.groupby(["corruption", "img_id"]).size()
valid_groups = group_sizes[group_sizes >= 3].index
df_css = df_css.set_index(["corruption", "img_id"])
df_css = df_css.loc[df_css.index.isin(valid_groups)].reset_index()

# FACTOR's actual formula: CSS = sigmoid(KL - per-image mean KL)
df_css["kl_mean_image"] = df_css.groupby(["corruption", "img_id"])["kl_avg"].transform("mean")
df_css["css"] = 1 / (1 + np.exp(-(df_css["kl_avg"] - df_css["kl_mean_image"])))

print(f"Detections retained (images with ≥3 detections): {len(df_css)}")
print()

# ─────────────────────────────────────────────────────────────
# Correlation check: score vs CSS (the relativized signal)
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("CORRELATION RESULTS — Relative CSS vs Original Score")
print("=" * 65)
sp, sp_p = spearmanr(df_css["score_orig"], df_css["css"])
pe, pe_p = pearsonr(df_css["score_orig"], df_css["css"])
print(f"score_orig vs css   Spearman: {sp:+.3f} (p={sp_p:.4f})  Pearson: {pe:+.3f} (p={pe_p:.4f})")
print(f"(Compare to RAW KL correlation: Spearman +0.375 — checking whether relativizing")
print(f" removes the confidence-dependent confound)")
print()

for corruption in DIAG_CORRUPTIONS:
    df_c = df_css[df_css["corruption"] == corruption]
    if len(df_c) < 3:
        continue
    sp, sp_p = spearmanr(df_c["score_orig"], df_c["css"])
    print(f"{corruption}: Spearman {sp:+.3f} (p={sp_p:.4f}), N={len(df_c)}")

print()
print("─" * 65)
print("CSS distribution summary (should span a meaningful 0-1 range,")
print("not collapse to a narrow band around 0.5)")
print("─" * 65)
print(df_css["css"].describe())

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_css.to_csv("/kaggle/working/tables/table_step2b_css_relative.csv", index=False)
print(f"\n✅ Saved: table_step2b_css_relative.csv")
print()
print("=" * 65)
print("RELATIVE GATING CHECK COMPLETE")
print("=" * 65)

In [ ]:
print(df_valid["kl_avg"].describe())

In [ ]:
# ============================================================
# Setup for Raw-Score Bernoulli KL Fix
# Consolidates the perturbation + config pieces from earlier
# diagnostics that we're still reusing — the softmax-based KL
# correlation-checking code itself is NOT reused, since we're
# replacing that divergence measure entirely.
# ============================================================

import numpy as np
import random as _py_random
from PIL import Image as PILImage
import torchvision.transforms.functional as TF
import torchvision.transforms as T
from imagecorruptions import corrupt as ic_corrupt

print("=" * 65)
print("Setup: Corruption Seeding + Strong Perturbation Views")
print("=" * 65)
print()

# ── Deterministic corruption seeding (unchanged from earlier blocks) ──
CORRUPTION_SEED_OFFSET = {
    "gaussian_noise": 1_000, "motion_blur": 2_000,
    "snow": 3_000, "contrast": 4_000,
}

def seed_for_corruption(img_id, corruption, severity):
    offset = CORRUPTION_SEED_OFFSET.get(corruption, 9_000)
    return (img_id * 100 + offset + severity) % (2**31)

def apply_corruption_deterministic(raw_img, img_id, corruption, severity):
    seed_val = seed_for_corruption(img_id, corruption, severity)
    np.random.seed(seed_val)
    _py_random.seed(seed_val)
    return ic_corrupt(raw_img, corruption_name=corruption, severity=severity)

# ── Six distinct, strong, single-attribute perturbations ──
def strong_view(image_np: np.ndarray, view_idx: int) -> np.ndarray:
    img = PILImage.fromarray(image_np.astype(np.uint8))

    if view_idx == 1:      # Strong brightness reduction
        img = TF.adjust_brightness(img, 0.45)
    elif view_idx == 2:    # Strong contrast reduction
        img = TF.adjust_contrast(img, 0.45)
    elif view_idx == 3:    # Strong blur
        img = T.GaussianBlur(kernel_size=9, sigma=2.5)(img)
    elif view_idx == 4:    # Strong additive noise
        arr = np.array(img).astype(np.float32)
        noise = np.random.RandomState(42).normal(0, 35, arr.shape)
        arr = np.clip(arr + noise, 0, 255).astype(np.uint8)
        img = PILImage.fromarray(arr)
    elif view_idx == 5:    # Strong saturation reduction
        img = TF.adjust_saturation(img, 0.25)
    elif view_idx == 6:    # Strong hue shift
        img = TF.adjust_hue(img, 0.15)

    return np.array(img)

VIEW_NAMES = {1: "brightness", 2: "contrast", 3: "blur",
              4: "noise", 5: "saturation", 6: "hue"}
N_STRONG_VIEWS = 6

# ── Diagnostic scope (same as all prior correlation checks) ──
DIAG_CORRUPTIONS = ["motion_blur", "contrast"]
DIAG_SEVERITY    = 5
DIAG_N           = 10

print(f"✅ Corruption seeding helpers defined")
print(f"✅ 6 strong single-attribute views defined: {list(VIEW_NAMES.values())}")
print(f"✅ Diagnostic scope: {DIAG_CORRUPTIONS}, severity={DIAG_SEVERITY}, N={DIAG_N}")
print()
print("=" * 65)
print("SETUP COMPLETE — ready for the raw-score Bernoulli KL fix")
print("=" * 65)

In [ ]:
# ============================================================
# Raw-Score Bernoulli KL Divergence
# ============================================================
import torch
import numpy as np
import pandas as pd
from torchvision.ops import box_iou
from scipy.stats import spearmanr, pearsonr
from tqdm.notebook import tqdm
import os

print("=" * 65)
print("Raw-Score Bernoulli KL Divergence — Correlation Check")
print("=" * 65)
print()

IOU_THR_KL = 0.40

# ─────────────────────────────────────────────────────────────
# Updated extraction: now ALSO returns raw (pre-softmax) scores,
# not just the softmax-normalized distribution
# ─────────────────────────────────────────────────────────────
def run_dino_with_categories_raw(image_np, box_threshold=0.25):
    dino_model.eval()
    with torch.no_grad():
        inputs = dino_processor(images=image_np, text=DINO_TEXT_PROMPT, return_tensors="pt").to(device)
        outputs = dino_model(**inputs)

    cat_scores_raw, cat_dist = get_category_distribution(outputs, category_token_spans, category_order)
    max_raw_scores, max_cat_idx = cat_scores_raw.max(dim=-1)

    keep_mask    = max_raw_scores >= box_threshold
    keep_indices = keep_mask.nonzero(as_tuple=True)[0]

    if len(keep_indices) == 0:
        return [], [], [], [], None, None

    img_h, img_w = image_np.shape[:2]
    pred_cxcywh = outputs.pred_boxes[0]
    cx = pred_cxcywh[:, 0] * img_w
    cy = pred_cxcywh[:, 1] * img_h
    pw = pred_cxcywh[:, 2] * img_w
    ph = pred_cxcywh[:, 3] * img_h
    pred_xyxy = torch.stack([cx - pw/2, cy - ph/2, cx + pw/2, cy + ph/2], dim=-1)

    boxes     = pred_xyxy[keep_indices]
    scores    = max_raw_scores[keep_indices]
    labels    = [category_order[i] for i in max_cat_idx[keep_indices].tolist()]
    query_idx = keep_indices.tolist()
    full_dist = cat_dist[keep_indices]
    raw_dist  = cat_scores_raw[keep_indices]   # ← NEW: raw per-category sigmoid scores

    return boxes, scores, labels, query_idx, raw_dist, full_dist

print("✅ run_dino_with_categories_raw defined (now also returns raw scores)")
print()

# ─────────────────────────────────────────────────────────────
# Bernoulli KL — treats each of the 80 categories as an
# independent binary judgment, matching how the model was
# actually trained (no forced cross-category normalization)
# ─────────────────────────────────────────────────────────────
def bernoulli_kl(p, q, eps=1e-6):
    p = p.clamp(eps, 1 - eps)
    q = q.clamp(eps, 1 - eps)
    kl = p * torch.log(p / q) + (1 - p) * torch.log((1 - p) / (1 - q))
    return kl.sum().item()

print("✅ bernoulli_kl defined")
print()

def raw_kl_records_for_image(c_img):
    boxes_ref, scores_ref, labels_ref, qidx_ref, raw_ref, dist_ref = run_dino_with_categories_raw(c_img)
    if len(boxes_ref) == 0:
        return []

    view_data = []
    for v in range(1, N_STRONG_VIEWS + 1):
        aug = strong_view(c_img, v)
        boxes_v, scores_v, labels_v, qidx_v, raw_v, dist_v = run_dino_with_categories_raw(aug)
        view_data.append((boxes_v, raw_v))

    records = []
    for i in range(len(boxes_ref)):
        ref_box = boxes_ref[i].unsqueeze(0)
        kls = []
        for (boxes_v, raw_v) in view_data:
            if len(boxes_v) == 0:
                continue
            ious = box_iou(ref_box, boxes_v)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= IOU_THR_KL:
                kl = bernoulli_kl(raw_ref[i], raw_v[best_j.item()])
                kls.append(kl)

        records.append({
            "score_orig": scores_ref[i].item(),
            "label": labels_ref[i],
            "kl_avg": float(np.mean(kls)) if kls else np.nan,
            "n_matched_views": len(kls),
        })
    return records

# ─────────────────────────────────────────────────────────────
# Run on the same N=10, 2 corruptions, severity 5
# ─────────────────────────────────────────────────────────────
det_records_rawkl = []

for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[:DIAG_N], desc=f"{corruption} sev{DIAG_SEVERITY}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        recs = raw_kl_records_for_image(c_img)
        for r in recs:
            r["corruption"] = corruption
            r["img_id"] = img_id
            det_records_rawkl.append(r)

df_rawkl = pd.DataFrame(det_records_rawkl)
print(f"\nTotal detections: {len(df_rawkl)}")
print(f"Detections with at least 1 matched view: {(df_rawkl['n_matched_views'] > 0).sum()}")
print()

df_valid_raw = df_rawkl.dropna(subset=["kl_avg"]).copy()

# ─────────────────────────────────────────────────────────────
# Magnitude check FIRST — does this actually have more headroom?
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("MAGNITUDE CHECK — raw KL values (compare to previous: mean 0.0002, max 0.0014)")
print("=" * 65)
print(df_valid_raw["kl_avg"].describe())
print()

# ─────────────────────────────────────────────────────────────
# Correlation check
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("CORRELATION RESULTS — Raw-Score Bernoulli KL vs Original Score")
print("=" * 65)
sp, sp_p = spearmanr(df_valid_raw["score_orig"], df_valid_raw["kl_avg"])
pe, pe_p = pearsonr(df_valid_raw["score_orig"], df_valid_raw["kl_avg"])
print(f"score_orig vs kl_avg   Spearman: {sp:+.3f} (p={sp_p:.4f})  Pearson: {pe:+.3f} (p={pe_p:.4f})")
print()

for corruption in DIAG_CORRUPTIONS:
    df_c = df_valid_raw[df_valid_raw["corruption"] == corruption]
    sp, sp_p = spearmanr(df_c["score_orig"], df_c["kl_avg"])
    print(f"{corruption}: Spearman {sp:+.3f} (p={sp_p:.4f}), N={len(df_c)}")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_rawkl.to_csv("/kaggle/working/tables/table_step3_rawkl.csv", index=False)
print(f"\n✅ Saved: table_step3_rawkl.csv")
print()
print("=" * 65)
print("RAW-SCORE BERNOULLI KL CHECK COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# First mAP Test: Raw-Score Bernoulli KL → Per-Image Relative
# Gating → Score Recalibration
# ============================================================
import torch
import numpy as np
import pandas as pd
from torchvision.ops import box_iou
from tqdm.notebook import tqdm
import os

print("=" * 65)
print("First mAP Test: KL-Based Recalibration vs Frozen Baseline")
print("=" * 65)
print()

LAMBDA = 1.0  # decay strength on the recalibration — starting point

def recal_records_for_image(c_img, img_id):
    boxes_ref, scores_ref, labels_ref, qidx_ref, raw_ref, dist_ref = run_dino_with_categories_raw(c_img)

    preds_base = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": s.item()}
        for b, s, l in zip(boxes_ref, scores_ref, labels_ref) if l in COCO_MAP
    ]
    mAP_base, _ = compute_map(preds_base, coco_gt, [img_id])

    if len(boxes_ref) == 0:
        return mAP_base, mAP_base, 0

    view_data = []
    for v in range(1, N_STRONG_VIEWS + 1):
        aug = strong_view(c_img, v)
        boxes_v, scores_v, labels_v, qidx_v, raw_v, dist_v = run_dino_with_categories_raw(aug)
        view_data.append((boxes_v, raw_v))

    kl_vals = []
    for i in range(len(boxes_ref)):
        ref_box = boxes_ref[i].unsqueeze(0)
        kls = []
        for (boxes_v, raw_v) in view_data:
            if len(boxes_v) == 0:
                continue
            ious = box_iou(ref_box, boxes_v)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= 0.40:
                kls.append(bernoulli_kl(raw_ref[i], raw_v[best_j.item()]))
        kl_vals.append(np.mean(kls) if kls else 0.0)

    kl_vals = np.array(kl_vals)
    mean_kl = kl_vals.mean()
    css = 1 / (1 + np.exp(-(kl_vals - mean_kl)))   # per-image relative gating

    recal_scores = [s.item() * np.exp(-LAMBDA * c) for s, c in zip(scores_ref, css)]

    preds_recal = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": rs}
        for b, rs, l in zip(boxes_ref, recal_scores, labels_ref) if l in COCO_MAP
    ]
    mAP_recal, _ = compute_map(preds_recal, coco_gt, [img_id])

    return mAP_base, mAP_recal, len(boxes_ref)

results = []
for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[:DIAG_N], desc=f"{corruption} sev{DIAG_SEVERITY}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        mAP_base, mAP_recal, n_dets = recal_records_for_image(c_img, img_id)
        results.append({
            "corruption": corruption, "img_id": img_id,
            "mAP_baseline": mAP_base, "mAP_recal": mAP_recal,
            "vs_baseline": mAP_recal - mAP_base,
            "beats_baseline": mAP_recal > mAP_base,
            "n_detections": n_dets,
        })

df_map = pd.DataFrame(results)
print()
print("=" * 65)
print("RESULTS")
print("=" * 65)
print(df_map.groupby("corruption")[["vs_baseline"]].mean())
print()
print(f"Overall vs_baseline: {df_map['vs_baseline'].mean():+.4f}")
print(f"Beats baseline: {df_map['beats_baseline'].sum()}/{len(df_map)}")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_map.to_csv("/kaggle/working/tables/table_step4_map_test.csv", index=False)
print("\n✅ Saved: table_step4_map_test.csv")

In [ ]:
import numpy as np

css_check = []
for corruption in DIAG_CORRUPTIONS:
    for img_path in image_files[:DIAG_N]:
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        boxes_ref, scores_ref, labels_ref, qidx_ref, raw_ref, dist_ref = run_dino_with_categories_raw(c_img)
        if len(boxes_ref) < 2:
            continue

        view_data = []
        for v in range(1, N_STRONG_VIEWS + 1):
            aug = strong_view(c_img, v)
            boxes_v, scores_v, labels_v, qidx_v, raw_v, dist_v = run_dino_with_categories_raw(aug)
            view_data.append((boxes_v, raw_v))

        kl_vals = []
        for i in range(len(boxes_ref)):
            ref_box = boxes_ref[i].unsqueeze(0)
            kls = []
            for (boxes_v, raw_v) in view_data:
                if len(boxes_v) == 0:
                    continue
                ious = box_iou(ref_box, boxes_v)
                max_iou, best_j = ious[0].max(0)
                if max_iou.item() >= 0.40:
                    kls.append(bernoulli_kl(raw_ref[i], raw_v[best_j.item()]))
            kl_vals.append(np.mean(kls) if kls else 0.0)

        kl_vals = np.array(kl_vals)
        css = 1 / (1 + np.exp(-(kl_vals - kl_vals.mean())))
        css_check.append({"corruption": corruption, "img_id": img_id,
                           "css_std": css.std(), "css_range": css.max()-css.min(),
                           "n_dets": len(boxes_ref)})

df_css_check = pd.DataFrame(css_check)
print(df_css_check.groupby("corruption")[["css_std", "css_range", "n_dets"]].mean())

In [ ]:
# ============================================================
# Second mAP Test: Direct Raw-KL Recalibration (no relativization)
# ============================================================
LAMBDA_RAW = 1.0

def recal_records_raw_kl(c_img, img_id):
    boxes_ref, scores_ref, labels_ref, qidx_ref, raw_ref, dist_ref = run_dino_with_categories_raw(c_img)

    preds_base = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": s.item()}
        for b, s, l in zip(boxes_ref, scores_ref, labels_ref) if l in COCO_MAP
    ]
    mAP_base, _ = compute_map(preds_base, coco_gt, [img_id])

    if len(boxes_ref) == 0:
        return mAP_base, mAP_base, 0

    view_data = []
    for v in range(1, N_STRONG_VIEWS + 1):
        aug = strong_view(c_img, v)
        boxes_v, scores_v, labels_v, qidx_v, raw_v, dist_v = run_dino_with_categories_raw(aug)
        view_data.append((boxes_v, raw_v))

    kl_vals = []
    for i in range(len(boxes_ref)):
        ref_box = boxes_ref[i].unsqueeze(0)
        kls = []
        for (boxes_v, raw_v) in view_data:
            if len(boxes_v) == 0:
                continue
            ious = box_iou(ref_box, boxes_v)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= 0.40:
                kls.append(bernoulli_kl(raw_ref[i], raw_v[best_j.item()]))
        kl_vals.append(np.mean(kls) if kls else 0.0)

    # NO per-image relativization — use raw KL directly
    recal_scores = [s.item() * np.exp(-LAMBDA_RAW * kl) for s, kl in zip(scores_ref, kl_vals)]

    preds_recal = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": rs}
        for b, rs, l in zip(boxes_ref, recal_scores, labels_ref) if l in COCO_MAP
    ]
    mAP_recal, _ = compute_map(preds_recal, coco_gt, [img_id])

    return mAP_base, mAP_recal, len(boxes_ref)

results2 = []
for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[:DIAG_N], desc=f"{corruption} sev{DIAG_SEVERITY}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        mAP_base, mAP_recal, n_dets = recal_records_raw_kl(c_img, img_id)
        results2.append({
            "corruption": corruption, "img_id": img_id,
            "mAP_baseline": mAP_base, "mAP_recal": mAP_recal,
            "vs_baseline": mAP_recal - mAP_base,
            "beats_baseline": mAP_recal > mAP_base,
            "n_detections": n_dets,
        })

df_map2 = pd.DataFrame(results2)
print()
print(df_map2.groupby("corruption")[["vs_baseline"]].mean())
print(f"\nOverall vs_baseline: {df_map2['vs_baseline'].mean():+.4f}")
print(f"Beats baseline: {df_map2['beats_baseline'].sum()}/{len(df_map2)}")

In [ ]:
# ============================================================
# Stage 1: Build the Empirical Attribute Vulnerability Matrix
# Runtime: ~90 seconds
# ============================================================
import numpy as np
import pandas as pd
from torchvision.ops import box_iou
from tqdm.notebook import tqdm
import os

print("=" * 65)
print("Stage 1: Empirical Attribute Vulnerability Matrix")
print("=" * 65)
print()

CALIB_START      = 10   # image_files[10:20] — distinct from test set image_files[:10]
CALIB_N          = 10
CALIB_CORRUPTIONS = ["motion_blur", "contrast"]
CALIB_SEVERITY   = 5
MIN_SAMPLES      = 2     # categories with fewer calibration examples fall back to global average

def per_attribute_kl_for_detection(c_img):
    """Per-detection KL divergence broken out by INDIVIDUAL attribute,
    not averaged — we need the per-attribute breakdown for calibration."""
    boxes_ref, scores_ref, labels_ref, qidx_ref, raw_ref, dist_ref = run_dino_with_categories_raw(c_img)
    if len(boxes_ref) == 0:
        return []

    view_kls = {}
    for v in range(1, N_STRONG_VIEWS + 1):
        aug = strong_view(c_img, v)
        boxes_v, scores_v, labels_v, qidx_v, raw_v, dist_v = run_dino_with_categories_raw(aug)
        attr_name = VIEW_NAMES[v]
        if len(boxes_v) == 0:
            continue
        for i in range(len(boxes_ref)):
            ref_box = boxes_ref[i].unsqueeze(0)
            ious = box_iou(ref_box, boxes_v)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= 0.40:
                kl = bernoulli_kl(raw_ref[i], raw_v[best_j.item()])
                view_kls.setdefault(i, {})[attr_name] = kl

    records = []
    for i in range(len(boxes_ref)):
        records.append({
            "category": labels_ref[i],
            "score": scores_ref[i].item(),
            "kl_per_attr": view_kls.get(i, {}),
        })
    return records

print("✅ per_attribute_kl_for_detection defined")
print()

# ─────────────────────────────────────────────────────────────
# Run calibration pass
# ─────────────────────────────────────────────────────────────
calib_records = []

for corruption in CALIB_CORRUPTIONS:
    for img_path in tqdm(image_files[CALIB_START:CALIB_START+CALIB_N],
                          desc=f"calibrating on {corruption}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, CALIB_SEVERITY)

        recs = per_attribute_kl_for_detection(c_img)
        for r in recs:
            for attr, kl in r["kl_per_attr"].items():
                calib_records.append({
                    "corruption": corruption,
                    "category": r["category"],
                    "attribute": attr,
                    "kl": kl,
                })

df_calib = pd.DataFrame(calib_records)
print(f"\nTotal calibration records: {len(df_calib)}")
print(f"Unique categories observed: {df_calib['category'].nunique()} / 80")
print()

# ─────────────────────────────────────────────────────────────
# Build the vulnerability matrix, with fallback for sparse categories
# ─────────────────────────────────────────────────────────────
vulnerability_matrix = {}
coverage_report = {}

for corruption in CALIB_CORRUPTIONS:
    df_c = df_calib[df_calib["corruption"] == corruption]
    global_avg = df_c.groupby("attribute")["kl"].mean().to_dict()
    global_fallback = df_c["kl"].mean()

    cat_attr_stats = df_c.groupby(["category", "attribute"])["kl"].agg(["mean", "count"]).reset_index()

    vmat = {}
    n_calibrated, n_fallback = 0, 0
    for cat in category_order:
        vmat[cat] = {}
        for attr in VIEW_NAMES.values():
            row = cat_attr_stats[(cat_attr_stats["category"] == cat) &
                                  (cat_attr_stats["attribute"] == attr)]
            if len(row) > 0 and row.iloc[0]["count"] >= MIN_SAMPLES:
                vmat[cat][attr] = row.iloc[0]["mean"]
                n_calibrated += 1
            else:
                vmat[cat][attr] = global_avg.get(attr, global_fallback)
                n_fallback += 1

    vulnerability_matrix[corruption] = vmat
    coverage_report[corruption] = (n_calibrated, n_fallback)

print("=" * 65)
print("Coverage Report")
print("=" * 65)
for corruption, (n_cal, n_fb) in coverage_report.items():
    total = n_cal + n_fb
    print(f"{corruption}: {n_cal}/{total} (category, attribute) cells empirically calibrated, "
          f"{n_fb}/{total} using global fallback")
print()

# ─────────────────────────────────────────────────────────────
# Sanity check: do a few categories show intuitive patterns?
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("Sanity check: vulnerability profiles for a few observed categories")
print("=" * 65)
observed_cats = df_calib["category"].value_counts().head(5).index.tolist()
for corruption in CALIB_CORRUPTIONS:
    print(f"\n--- {corruption} ---")
    for cat in observed_cats:
        profile = vulnerability_matrix[corruption][cat]
        sorted_attrs = sorted(profile.items(), key=lambda x: -x[1])
        top3 = ", ".join(f"{a}={v:.3f}" for a, v in sorted_attrs[:3])
        print(f"  {cat:15s} most vulnerable to: {top3}")

os.makedirs("/kaggle/working/results", exist_ok=True)
import json
with open("/kaggle/working/results/vulnerability_matrix.json", "w") as f:
    json.dump(vulnerability_matrix, f, indent=2)
df_calib.to_csv("/kaggle/working/tables/table_calibration_raw.csv", index=False)

print()
print("✅ Saved: vulnerability_matrix.json, table_calibration_raw.csv")
print()
print("=" * 65)
print("STAGE 1 COMPLETE")
print("=" * 65)

In [ ]:
# Expand image pool to at least 70 (10 test + 60 calibration, non-overlapping)
TARGET_N = 70

print(f"Current image pool size: {len(image_files)}")

if len(image_files) < TARGET_N:
    IMAGE_DIR_CURRENT = os.path.dirname(image_files[0])
    all_candidates = sorted(
        f for f in os.listdir(IMAGE_DIR_CURRENT) if f.lower().endswith(".jpg")
    )
    existing_basenames = {os.path.basename(p) for p in image_files}

    added = 0
    for fname in all_candidates:
        if len(image_files) >= TARGET_N:
            break
        if fname in existing_basenames:
            continue
        full_path = os.path.join(IMAGE_DIR_CURRENT, fname)
        try:
            candidate_img_id = int(os.path.splitext(fname)[0])
        except ValueError:
            continue
        ann_ids = coco_gt.getAnnIds(imgIds=[candidate_img_id])
        if len(ann_ids) == 0:
            continue
        img_arr = np.array(PILImage.open(full_path).convert("RGB"))
        loaded_images[full_path] = img_arr
        img_id_map[fname] = candidate_img_id
        image_files.append(full_path)
        added += 1

    print(f"✅ Added {added} images — pool now at {len(image_files)}")
else:
    print("✅ Already have enough images")

In [ ]:
# ============================================================
# Stage 1b: Re-run Calibration at N=50
# Runtime: ~7-8 minutes
# ============================================================
CALIB_START      = 10   # image_files[10:60] — still distinct from test set [:10]
CALIB_N          = 50
CALIB_CORRUPTIONS = ["motion_blur", "contrast"]
CALIB_SEVERITY   = 5
MIN_SAMPLES      = 2

calib_records = []

for corruption in CALIB_CORRUPTIONS:
    for img_path in tqdm(image_files[CALIB_START:CALIB_START+CALIB_N],
                          desc=f"calibrating on {corruption}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, CALIB_SEVERITY)

        recs = per_attribute_kl_for_detection(c_img)
        for r in recs:
            for attr, kl in r["kl_per_attr"].items():
                calib_records.append({
                    "corruption": corruption,
                    "category": r["category"],
                    "attribute": attr,
                    "kl": kl,
                })

df_calib = pd.DataFrame(calib_records)
print(f"\nTotal calibration records: {len(df_calib)}")
print(f"Unique categories observed: {df_calib['category'].nunique()} / 80")
print()

vulnerability_matrix = {}
coverage_report = {}

for corruption in CALIB_CORRUPTIONS:
    df_c = df_calib[df_calib["corruption"] == corruption]
    global_avg = df_c.groupby("attribute")["kl"].mean().to_dict()
    global_fallback = df_c["kl"].mean()

    cat_attr_stats = df_c.groupby(["category", "attribute"])["kl"].agg(["mean", "count"]).reset_index()

    vmat = {}
    n_calibrated, n_fallback = 0, 0
    for cat in category_order:
        vmat[cat] = {}
        for attr in VIEW_NAMES.values():
            row = cat_attr_stats[(cat_attr_stats["category"] == cat) &
                                  (cat_attr_stats["attribute"] == attr)]
            if len(row) > 0 and row.iloc[0]["count"] >= MIN_SAMPLES:
                vmat[cat][attr] = row.iloc[0]["mean"]
                n_calibrated += 1
            else:
                vmat[cat][attr] = global_avg.get(attr, global_fallback)
                n_fallback += 1

    vulnerability_matrix[corruption] = vmat
    coverage_report[corruption] = (n_calibrated, n_fallback)

print("=" * 65)
print("Coverage Report (N=50)")
print("=" * 65)
for corruption, (n_cal, n_fb) in coverage_report.items():
    total = n_cal + n_fb
    print(f"{corruption}: {n_cal}/{total} cells empirically calibrated, {n_fb}/{total} fallback")

os.makedirs("/kaggle/working/results", exist_ok=True)
os.makedirs("/kaggle/working/tables", exist_ok=True)
import json
with open("/kaggle/working/results/vulnerability_matrix_n50.json", "w") as f:
    json.dump(vulnerability_matrix, f, indent=2)
df_calib.to_csv("/kaggle/working/tables/table_calibration_n50.csv", index=False)
print("\n✅ Saved: vulnerability_matrix_n50.json, table_calibration_n50.csv")

In [ ]:
# ============================================================
# Stage 2: Compute "Surprise" Signal on Test Images
# surprise = observed KL - calibrated expected vulnerability
# Runtime: ~85-90 seconds
# ============================================================
import numpy as np
import pandas as pd
from scipy.stats import spearmanr, pearsonr
from tqdm.notebook import tqdm
import os

print("=" * 65)
print("Stage 2: Surprise Signal — Correlation Check")
print("=" * 65)
print()

surprise_records = []

for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[:DIAG_N], desc=f"{corruption} sev{DIAG_SEVERITY}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        recs = per_attribute_kl_for_detection(c_img)
        for r in recs:
            cat = r["category"]
            kl_per_attr = r["kl_per_attr"]
            if not kl_per_attr:
                continue

            vmat_cat = vulnerability_matrix[corruption].get(cat, {})
            surprises = [
                kl_per_attr[attr] - vmat_cat.get(attr, 0.0)
                for attr in kl_per_attr
            ]

            surprise_records.append({
                "corruption": corruption, "img_id": img_id, "category": cat,
                "score_orig": r["score"],
                "surprise": float(np.mean(surprises)),
                "n_attrs_observed": len(kl_per_attr),
            })

df_surprise = pd.DataFrame(surprise_records)
print(f"\nTotal detections: {len(df_surprise)}")
print()

# ─────────────────────────────────────────────────────────────
# Magnitude check
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("MAGNITUDE CHECK — surprise values")
print("=" * 65)
print(df_surprise["surprise"].describe())
print()

# ─────────────────────────────────────────────────────────────
# Correlation check
# ─────────────────────────────────────────────────────────────
print("=" * 65)
print("CORRELATION RESULTS — Surprise vs Original Score")
print("=" * 65)
sp, sp_p = spearmanr(df_surprise["score_orig"], df_surprise["surprise"])
pe, pe_p = pearsonr(df_surprise["score_orig"], df_surprise["surprise"])
print(f"score_orig vs surprise   Spearman: {sp:+.3f} (p={sp_p:.4f})  Pearson: {pe:+.3f} (p={pe_p:.4f})")
print()

for corruption in DIAG_CORRUPTIONS:
    df_c = df_surprise[df_surprise["corruption"] == corruption]
    sp, sp_p = spearmanr(df_c["score_orig"], df_c["surprise"])
    print(f"{corruption}: Spearman {sp:+.3f} (p={sp_p:.4f}), N={len(df_c)}")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_surprise.to_csv("/kaggle/working/tables/table_stage2_surprise.csv", index=False)
print(f"\n✅ Saved: table_stage2_surprise.csv")
print()
print("=" * 65)
print("STAGE 2 COMPLETE")
print("=" * 65)

In [ ]:
# ============================================================
# Stage 3: First mAP Test Using the Surprise Signal
# Runtime: ~85-90 seconds
# ============================================================
import torch
import numpy as np
import pandas as pd
from torchvision.ops import box_iou
from tqdm.notebook import tqdm
import os

LAMBDA_SURPRISE = 1.0

def surprise_recal_for_image(c_img, img_id, corruption):
    boxes_ref, scores_ref, labels_ref, qidx_ref, raw_ref, dist_ref = run_dino_with_categories_raw(c_img)

    preds_base = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": s.item()}
        for b, s, l in zip(boxes_ref, scores_ref, labels_ref) if l in COCO_MAP
    ]
    mAP_base, _ = compute_map(preds_base, coco_gt, [img_id])

    if len(boxes_ref) == 0:
        return mAP_base, mAP_base, 0

    view_data = []
    for v in range(1, N_STRONG_VIEWS + 1):
        aug = strong_view(c_img, v)
        boxes_v, scores_v, labels_v, qidx_v, raw_v, dist_v = run_dino_with_categories_raw(aug)
        view_data.append((VIEW_NAMES[v], boxes_v, raw_v))

    recal_scores = []
    for i in range(len(boxes_ref)):
        ref_box = boxes_ref[i].unsqueeze(0)
        kl_per_attr = {}
        for (attr_name, boxes_v, raw_v) in view_data:
            if len(boxes_v) == 0:
                continue
            ious = box_iou(ref_box, boxes_v)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= 0.40:
                kl_per_attr[attr_name] = bernoulli_kl(raw_ref[i], raw_v[best_j.item()])

        cat = labels_ref[i]
        vmat_cat = vulnerability_matrix[corruption].get(cat, {})
        if kl_per_attr:
            surprises = [kl_per_attr[a] - vmat_cat.get(a, 0.0) for a in kl_per_attr]
            surprise = float(np.mean(surprises))
        else:
            surprise = 0.0

        recal = scores_ref[i].item() * np.exp(-LAMBDA_SURPRISE * max(surprise, 0.0))
        recal_scores.append(recal)

    preds_recal = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": rs}
        for b, rs, l in zip(boxes_ref, recal_scores, labels_ref) if l in COCO_MAP
    ]
    mAP_recal, _ = compute_map(preds_recal, coco_gt, [img_id])

    return mAP_base, mAP_recal, len(boxes_ref)

results3 = []
for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[:DIAG_N], desc=f"{corruption} sev{DIAG_SEVERITY}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        mAP_base, mAP_recal, n_dets = surprise_recal_for_image(c_img, img_id, corruption)
        results3.append({
            "corruption": corruption, "img_id": img_id,
            "mAP_baseline": mAP_base, "mAP_recal": mAP_recal,
            "vs_baseline": mAP_recal - mAP_base,
            "beats_baseline": mAP_recal > mAP_base,
            "n_detections": n_dets,
        })

df_map3 = pd.DataFrame(results3)
print()
print(df_map3.groupby("corruption")[["vs_baseline"]].mean())
print(f"\nOverall vs_baseline: {df_map3['vs_baseline'].mean():+.4f}")
print(f"Beats baseline: {df_map3['beats_baseline'].sum()}/{len(df_map3)}")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_map3.to_csv("/kaggle/working/tables/table_stage3_map_test.csv", index=False)
print("\n✅ Saved: table_stage3_map_test.csv")

In [ ]:
# ============================================================
# Online Memory Mechanism — Part 1: Structure + Warm-Start
# Runtime: <1 second (no model inference, just loading saved data)
# ============================================================
import pandas as pd
import os

print("=" * 65)
print("Online Memory: Welford Running Stats, Warm-Started from Calibration")
print("=" * 65)
print()

class RunningStats:
    """Numerically stable online mean/variance via Welford's algorithm —
    lets us compute a proper z-score (mean AND variance), not just a
    raw difference from a fixed mean, and lets the estimate keep
    improving as more evidence streams in."""
    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self.M2 = 0.0

    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.M2 += delta * delta2

    @property
    def variance(self):
        return self.M2 / self.n if self.n > 1 else 0.0

    @property
    def std(self):
        return self.variance ** 0.5

    def zscore(self, x, eps=1e-6):
        return (x - self.mean) / (self.std + eps)

# ── Memory: nested dict [corruption][category][attribute] -> RunningStats ──
memory = {}

# ── Warm-start from Stage 1b's saved calibration data ──
calib_path = "/kaggle/working/tables/table_calibration_n50.csv"

if os.path.exists(calib_path):
    df_warm = pd.read_csv(calib_path)
    print(f"✅ Found saved calibration data: {len(df_warm)} records")

    for _, row in df_warm.iterrows():
        c, cat, attr, kl = row["corruption"], row["category"], row["attribute"], row["kl"]
        memory.setdefault(c, {}).setdefault(cat, {}).setdefault(attr, RunningStats())
        memory[c][cat][attr].update(kl)

    n_cells = sum(len(cats) * len(attrs) for c in memory.values()
                  for cats in [c] for attrs in [next(iter(c.values())).keys() if c else []])
    n_cat_attr_pairs = sum(len(attrs) for cats in memory.values() for attrs in cats.values())
    print(f"✅ Memory warm-started: {n_cat_attr_pairs} (category, attribute) cells initialised")
    for c in memory:
        print(f"   {c}: {len(memory[c])} categories with real history")
else:
    print("⚠️  No saved calibration file found — memory will start cold (empty)")
    print("   This is fine, but expect the first several test images to fall back")
    print("   heavily to global defaults until enough history accumulates.")

print()
print("=" * 65)
print("MEMORY STRUCTURE READY")
print("=" * 65)

In [ ]:
TARGET_N = 70

print(f"Current image pool size: {len(image_files)}")

if len(image_files) < TARGET_N:
    IMAGE_DIR_CURRENT = os.path.dirname(image_files[0])
    all_candidates = sorted(
        f for f in os.listdir(IMAGE_DIR_CURRENT) if f.lower().endswith(".jpg")
    )
    existing_basenames = {os.path.basename(p) for p in image_files}

    added = 0
    for fname in all_candidates:
        if len(image_files) >= TARGET_N:
            break
        if fname in existing_basenames:
            continue
        full_path = os.path.join(IMAGE_DIR_CURRENT, fname)
        try:
            candidate_img_id = int(os.path.splitext(fname)[0])
        except ValueError:
            continue
        ann_ids = coco_gt.getAnnIds(imgIds=[candidate_img_id])
        if len(ann_ids) == 0:
            continue
        img_arr = np.array(PILImage.open(full_path).convert("RGB"))
        loaded_images[full_path] = img_arr
        img_id_map[fname] = candidate_img_id
        image_files.append(full_path)
        added += 1

    print(f"✅ Added {added} images — pool now at {len(image_files)}")
else:
    print("✅ Already have enough images")

In [ ]:
CALIB_START      = 10
CALIB_N          = 50
CALIB_CORRUPTIONS = ["motion_blur", "contrast"]
CALIB_SEVERITY   = 5

calib_records = []

for corruption in CALIB_CORRUPTIONS:
    for img_path in tqdm(image_files[CALIB_START:CALIB_START+CALIB_N],
                          desc=f"calibrating on {corruption}"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, CALIB_SEVERITY)

        recs = per_attribute_kl_for_detection(c_img)
        for r in recs:
            for attr, kl in r["kl_per_attr"].items():
                calib_records.append({
                    "corruption": corruption,
                    "category": r["category"],
                    "attribute": attr,
                    "kl": kl,
                })

df_calib = pd.DataFrame(calib_records)
print(f"\nTotal calibration records: {len(df_calib)}")
print(f"Unique categories observed: {df_calib['category'].nunique()} / 80")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_calib.to_csv("/kaggle/working/tables/table_calibration_n50.csv", index=False)
print("✅ Saved: table_calibration_n50.csv")

In [ ]:
# ============================================================
# Online Memory Mechanism — Sequential Test + mAP
# Runtime: ~85-90 seconds
# ============================================================
import numpy as np
import pandas as pd
from torchvision.ops import box_iou
from tqdm.notebook import tqdm
import os

print("=" * 65)
print("Online Memory Mechanism — Sequential Test + mAP")
print("=" * 65)
print()

class RunningStats:
    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self.M2 = 0.0

    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.M2 += delta * delta2

    @property
    def variance(self):
        return self.M2 / self.n if self.n > 1 else 0.0

    @property
    def std(self):
        return self.variance ** 0.5

    def zscore(self, x, eps=1e-6):
        return (x - self.mean) / (self.std + eps)

# ── Warm-start memory + global fallback from saved calibration ──
memory = {}
global_stats = {}

calib_path = "/kaggle/working/tables/table_calibration_n50.csv"
df_warm = pd.read_csv(calib_path)
for _, row in df_warm.iterrows():
    c, cat, attr, kl = row["corruption"], row["category"], row["attribute"], row["kl"]
    memory.setdefault(c, {}).setdefault(cat, {}).setdefault(attr, RunningStats()).update(kl)
    global_stats.setdefault(c, RunningStats()).update(kl)

print(f"✅ Memory warm-started from {len(df_warm)} calibration records")
print()

def get_stats(corruption, category, attribute, min_n=2):
    cell = memory.get(corruption, {}).get(category, {}).get(attribute)
    if cell is not None and cell.n >= min_n:
        return cell
    return global_stats.setdefault(corruption, RunningStats())

LAMBDA_MEM = 1.0

def memory_recal_for_image(c_img, img_id, corruption):
    boxes_ref, scores_ref, labels_ref, qidx_ref, raw_ref, dist_ref = run_dino_with_categories_raw(c_img)

    preds_base = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": s.item()}
        for b, s, l in zip(boxes_ref, scores_ref, labels_ref) if l in COCO_MAP
    ]
    mAP_base, _ = compute_map(preds_base, coco_gt, [img_id])

    if len(boxes_ref) == 0:
        return mAP_base, mAP_base, 0

    view_data = []
    for v in range(1, N_STRONG_VIEWS + 1):
        aug = strong_view(c_img, v)
        boxes_v, scores_v, labels_v, qidx_v, raw_v, dist_v = run_dino_with_categories_raw(aug)
        view_data.append((VIEW_NAMES[v], boxes_v, raw_v))

    recal_scores = []
    detection_kls = []

    for i in range(len(boxes_ref)):
        ref_box = boxes_ref[i].unsqueeze(0)
        kl_per_attr = {}
        for (attr_name, boxes_v, raw_v) in view_data:
            if len(boxes_v) == 0:
                continue
            ious = box_iou(ref_box, boxes_v)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= 0.40:
                kl_per_attr[attr_name] = bernoulli_kl(raw_ref[i], raw_v[best_j.item()])

        cat = labels_ref[i]
        zscores = []
        for attr, kl in kl_per_attr.items():
            stats = get_stats(corruption, cat, attr)
            zscores.append(stats.zscore(kl))
            detection_kls.append((cat, attr, kl))

        surprise_z = float(np.mean(zscores)) if zscores else 0.0
        recal = scores_ref[i].item() * np.exp(-LAMBDA_MEM * max(surprise_z, 0.0))
        recal_scores.append(recal)

    preds_recal = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": rs}
        for b, rs, l in zip(boxes_ref, recal_scores, labels_ref) if l in COCO_MAP
    ]
    mAP_recal, _ = compute_map(preds_recal, coco_gt, [img_id])

    # ── Update memory AFTER scoring — genuine online accumulation ──
    for cat, attr, kl in detection_kls:
        memory.setdefault(corruption, {}).setdefault(cat, {}).setdefault(attr, RunningStats()).update(kl)
        global_stats.setdefault(corruption, RunningStats()).update(kl)

    return mAP_base, mAP_recal, len(boxes_ref)

results_mem = []
for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[:DIAG_N], desc=f"{corruption} sev{DIAG_SEVERITY} (online)"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        mAP_base, mAP_recal, n_dets = memory_recal_for_image(c_img, img_id, corruption)
        results_mem.append({
            "corruption": corruption, "img_id": img_id,
            "mAP_baseline": mAP_base, "mAP_recal": mAP_recal,
            "vs_baseline": mAP_recal - mAP_base,
            "beats_baseline": mAP_recal > mAP_base,
            "n_detections": n_dets,
        })

df_mem = pd.DataFrame(results_mem)
print()
print(df_mem.groupby("corruption")[["vs_baseline"]].mean())
print(f"\nOverall vs_baseline: {df_mem['vs_baseline'].mean():+.4f}")
print(f"Beats baseline: {df_mem['beats_baseline'].sum()}/{len(df_mem)}")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_mem.to_csv("/kaggle/working/tables/table_online_memory_test.csv", index=False)
print("\n✅ Saved: table_online_memory_test.csv")

In [ ]:
# Expand pool further if needed, then validate on a genuinely held-out set
TARGET_N = 80
print(f"Current image pool size: {len(image_files)}")

if len(image_files) < TARGET_N:
    IMAGE_DIR_CURRENT = os.path.dirname(image_files[0])
    all_candidates = sorted(
        f for f in os.listdir(IMAGE_DIR_CURRENT) if f.lower().endswith(".jpg")
    )
    existing_basenames = {os.path.basename(p) for p in image_files}
    added = 0
    for fname in all_candidates:
        if len(image_files) >= TARGET_N:
            break
        if fname in existing_basenames:
            continue
        full_path = os.path.join(IMAGE_DIR_CURRENT, fname)
        try:
            candidate_img_id = int(os.path.splitext(fname)[0])
        except ValueError:
            continue
        if len(coco_gt.getAnnIds(imgIds=[candidate_img_id])) == 0:
            continue
        img_arr = np.array(PILImage.open(full_path).convert("RGB"))
        loaded_images[full_path] = img_arr
        img_id_map[fname] = candidate_img_id
        image_files.append(full_path)
        added += 1
    print(f"✅ Added {added} images — pool now at {len(image_files)}")

In [ ]:
# ============================================================
# Held-Out Validation: Online Memory on Genuinely Fresh Images
# Runtime: ~2.5-3 minutes
# ============================================================
import numpy as np
import pandas as pd
from torchvision.ops import box_iou
from tqdm.notebook import tqdm
import os

print("=" * 65)
print("Held-Out Validation — image_files[60:80], never used before")
print("=" * 65)
print()

class RunningStats:
    def __init__(self):
        self.n = 0
        self.mean = 0.0
        self.M2 = 0.0

    def update(self, x):
        self.n += 1
        delta = x - self.mean
        self.mean += delta / self.n
        delta2 = x - self.mean
        self.M2 += delta * delta2

    @property
    def variance(self):
        return self.M2 / self.n if self.n > 1 else 0.0

    @property
    def std(self):
        return self.variance ** 0.5

    def zscore(self, x, eps=1e-6):
        return (x - self.mean) / (self.std + eps)

# ── Fresh warm-start — reset to calibration-only state, no carryover
# from the previous test run on image_files[:10] ──
memory = {}
global_stats = {}

calib_path = "/kaggle/working/tables/table_calibration_n50.csv"
df_warm = pd.read_csv(calib_path)
for _, row in df_warm.iterrows():
    c, cat, attr, kl = row["corruption"], row["category"], row["attribute"], row["kl"]
    memory.setdefault(c, {}).setdefault(cat, {}).setdefault(attr, RunningStats()).update(kl)
    global_stats.setdefault(c, RunningStats()).update(kl)

print(f"✅ Memory reset and warm-started fresh from {len(df_warm)} calibration records")
print()

def get_stats(corruption, category, attribute, min_n=2):
    cell = memory.get(corruption, {}).get(category, {}).get(attribute)
    if cell is not None and cell.n >= min_n:
        return cell
    return global_stats.setdefault(corruption, RunningStats())

LAMBDA_MEM = 1.0

def memory_recal_for_image(c_img, img_id, corruption):
    boxes_ref, scores_ref, labels_ref, qidx_ref, raw_ref, dist_ref = run_dino_with_categories_raw(c_img)

    preds_base = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": s.item()}
        for b, s, l in zip(boxes_ref, scores_ref, labels_ref) if l in COCO_MAP
    ]
    mAP_base, _ = compute_map(preds_base, coco_gt, [img_id])

    if len(boxes_ref) == 0:
        return mAP_base, mAP_base, 0

    view_data = []
    for v in range(1, N_STRONG_VIEWS + 1):
        aug = strong_view(c_img, v)
        boxes_v, scores_v, labels_v, qidx_v, raw_v, dist_v = run_dino_with_categories_raw(aug)
        view_data.append((VIEW_NAMES[v], boxes_v, raw_v))

    recal_scores = []
    detection_kls = []

    for i in range(len(boxes_ref)):
        ref_box = boxes_ref[i].unsqueeze(0)
        kl_per_attr = {}
        for (attr_name, boxes_v, raw_v) in view_data:
            if len(boxes_v) == 0:
                continue
            ious = box_iou(ref_box, boxes_v)
            max_iou, best_j = ious[0].max(0)
            if max_iou.item() >= 0.40:
                kl_per_attr[attr_name] = bernoulli_kl(raw_ref[i], raw_v[best_j.item()])

        cat = labels_ref[i]
        zscores = []
        for attr, kl in kl_per_attr.items():
            stats = get_stats(corruption, cat, attr)
            zscores.append(stats.zscore(kl))
            detection_kls.append((cat, attr, kl))

        surprise_z = float(np.mean(zscores)) if zscores else 0.0
        recal = scores_ref[i].item() * np.exp(-LAMBDA_MEM * max(surprise_z, 0.0))
        recal_scores.append(recal)

    preds_recal = [
        {"image_id": img_id, "category_id": COCO_MAP[l],
         "bbox": [b[0].item(), b[1].item(), (b[2]-b[0]).item(), (b[3]-b[1]).item()],
         "score": rs}
        for b, rs, l in zip(boxes_ref, recal_scores, labels_ref) if l in COCO_MAP
    ]
    mAP_recal, _ = compute_map(preds_recal, coco_gt, [img_id])

    for cat, attr, kl in detection_kls:
        memory.setdefault(corruption, {}).setdefault(cat, {}).setdefault(attr, RunningStats()).update(kl)
        global_stats.setdefault(corruption, RunningStats()).update(kl)

    return mAP_base, mAP_recal, len(boxes_ref)

HOLDOUT_START = 60
HOLDOUT_N = 20

results_holdout = []
for corruption in DIAG_CORRUPTIONS:
    for img_path in tqdm(image_files[HOLDOUT_START:HOLDOUT_START+HOLDOUT_N],
                          desc=f"{corruption} sev{DIAG_SEVERITY} (held-out)"):
        img_id  = img_id_map[os.path.basename(img_path)]
        raw_img = loaded_images[img_path]
        c_img   = apply_corruption_deterministic(raw_img, img_id, corruption, DIAG_SEVERITY)

        mAP_base, mAP_recal, n_dets = memory_recal_for_image(c_img, img_id, corruption)
        results_holdout.append({
            "corruption": corruption, "img_id": img_id,
            "mAP_baseline": mAP_base, "mAP_recal": mAP_recal,
            "vs_baseline": mAP_recal - mAP_base,
            "beats_baseline": mAP_recal > mAP_base,
            "n_detections": n_dets,
        })

df_holdout = pd.DataFrame(results_holdout)
print()
print(df_holdout.groupby("corruption")[["vs_baseline"]].mean())
print(f"\nOverall vs_baseline: {df_holdout['vs_baseline'].mean():+.4f}")
print(f"Beats baseline: {df_holdout['beats_baseline'].sum()}/{len(df_holdout)}")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_holdout.to_csv("/kaggle/working/tables/table_holdout_validation.csv", index=False)
print("\n✅ Saved: table_holdout_validation.csv")

In [6]:
# ============================================================
# BLOCK 14: LoRA Adapter Injection — RANK 16 (MODIFIED)
# Injects trainable LoRA matrices into GroundingDINO's
# cross-modal neck (encoder + decoder query/value projections).
# All backbone weights remain frozen.
# Only LoRA matrices A and B are trainable.
# Reference: Hu et al. (2021) "LoRA: Low-Rank Adaptation"
# ============================================================
#
# WHY RANK=16 INSTEAD OF RANK=4
# ──────────────────────────────
# Block 24e (N=200, formally powered) showed vs_baseline = -0.0055
# (p<0.0001) with rank=4. Subsequent diagnostic (Block 24f) showed
# that switching from confidence-only to a full detection loss
# (L1 + GIoU + classification) produced IDENTICAL results —
# bit-for-bit, to 4 decimal places — confirming the loss function
# is NOT the bottleneck. The consistent LoRA Δ ≈ 0.0005 across all
# variants points to adaptation CAPACITY as the limiting factor:
# rank=4 (73,728 params) compresses all gradient signals —
# regardless of their source — into such a tiny weight perturbation
# that the model's predictions barely shift from the frozen baseline.
#
# RANK=16 gives 4× more capacity (≈294,912 params) without jumping
# to rank=32 (which risks overfitting to the sparse 1-2 verified
# detections per image). LORA_ALPHA is scaled proportionally
# (8→32) to maintain the same effective scale ratio (alpha/rank=2.0)
# as the original design — keeping the contribution magnitude
# consistent with the original intent.
#
# ONLY TWO LINES CHANGED vs the original Block 14:
#   LORA_RANK  : 4  → 16
#   LORA_ALPHA : 8  → 32
# Everything else is identical.
# ============================================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import json
import os

# --- 14.1 LoRA Linear Layer ---
class LoRALinear(nn.Module):
    """
    Replaces a Linear layer with a LoRA-augmented version.

    Weight update: W' = W + (B @ A) * scale
    Where:
        A : [rank, in_features]  — down-projection (Kaiming init)
        B : [out_features, rank] — up-projection   (zero init)
        scale = lora_alpha / rank

    B=0 at initialisation ensures no change to model output
    at the start of TTT. Only A and B are trainable.
    """
    def __init__(self, linear_layer, rank=4, lora_alpha=8):
        super().__init__()

        self.in_features  = linear_layer.in_features
        self.out_features = linear_layer.out_features
        self.rank         = rank
        self.scale        = lora_alpha / rank

        # Frozen original weights
        self.weight = nn.Parameter(
            linear_layer.weight.data.clone(),
            requires_grad=False
        )
        self.bias = nn.Parameter(
            linear_layer.bias.data.clone(),
            requires_grad=False
        ) if linear_layer.bias is not None else None

        # Trainable LoRA matrices
        self.lora_A = nn.Parameter(
            torch.zeros(rank, self.in_features)
        )
        self.lora_B = nn.Parameter(
            torch.zeros(self.out_features, rank)
        )

        # Kaiming init for A, zeros for B
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base  = nn.functional.linear(x, self.weight, self.bias)
        delta = (x @ self.lora_A.T @ self.lora_B.T) * self.scale
        return base + delta

    def extra_repr(self):
        return (f"in={self.in_features}, out={self.out_features}, "
                f"rank={self.rank}, scale={self.scale:.2f}")


# --- 14.2 LoRA Injection for GroundingDINO ---
def inject_lora_grounding_dino(model, rank=4, lora_alpha=8):
    """
    Injects LoRA into query and value projections of
    GroundingDINO's cross-modal neck only.

    Targeted components:
      - model.encoder: text enhancer + fusion attention layers
      - model.decoder: self_attn + encoder_attn_text layers

    Skipped components (frozen throughout):
      - model.backbone (vision encoder)
      - model.text_backbone (BERT text encoder)
    """
    n_injected = 0

    for name, module in list(model.named_modules()):

        if not (name.startswith('model.encoder') or
                name.startswith('model.decoder')):
            continue

        if not (name.endswith('.query') or
                name.endswith('.value')):
            continue

        if not isinstance(module, nn.Linear):
            continue

        parts  = name.split('.')
        parent = model
        for part in parts[:-1]:
            parent = getattr(parent, part)

        lora_layer = LoRALinear(
            module, rank=rank, lora_alpha=lora_alpha
        ).to(device)
        setattr(parent, parts[-1], lora_layer)
        n_injected += 1

    for param in model.parameters():
        param.requires_grad = False

    lora_params = []
    for name, param in model.named_parameters():
        if 'lora_A' in name or 'lora_B' in name:
            param.requires_grad = True
            lora_params.append((name, param))

    return n_injected, lora_params


# --- 14.3 Inject ---
LORA_RANK  = 4   # ← CHANGED from 4  (4× more capacity)
LORA_ALPHA = 8   # ← CHANGED from 8  (maintains scale ratio: 32/16 = 2.0)

print("Injecting LoRA adapters into GroundingDINO...")
print(f"Rank      : {LORA_RANK}  (was 4)")
print(f"Alpha     : {LORA_ALPHA}  (was 8)")
print(f"Scale     : {LORA_ALPHA / LORA_RANK:.2f}  (unchanged — same as rank=4 design)")
print(f"Targets   : query, value (encoder + decoder only)")
print()

n_injected, lora_params = inject_lora_grounding_dino(
    dino_model, rank=LORA_RANK, lora_alpha=LORA_ALPHA
)

print(f"✅ LoRA layers injected : {n_injected}")
print(f"✅ Trainable LoRA params: {len(lora_params)}")

# --- 14.4 Parameter Audit ---
total_params     = sum(p.numel() for p in dino_model.parameters())
trainable_params = sum(
    p.numel() for p in dino_model.parameters()
    if p.requires_grad
)
frozen_params = total_params - trainable_params

print(f"\n--- Parameter Audit ---")
print(f"Total parameters    : {total_params:,}")
print(f"Trainable (LoRA)    : {trainable_params:,}")
print(f"Frozen (backbone)   : {frozen_params:,}")
print(f"Trainable ratio     : "
      f"{100 * trainable_params / total_params:.4f}%")
print(f"vs rank=4 baseline  : {trainable_params / 73728:.1f}× more capacity")

assert trainable_params > 0, \
    "No trainable parameters — injection failed"
assert trainable_params < total_params * 0.05, \
    "Too many trainable params — check injection scope"
print(f"✅ Parameter audit passed")

# --- 14.5 Forward Pass Verification ---
print(f"\nVerifying forward pass after LoRA injection...")
test_img = loaded_images[image_files[0]]

inputs = dino_processor(
    images=test_img,
    text=DINO_TEXT_PROMPT,
    return_tensors="pt"
).to(device)

with torch.no_grad():
    outputs = dino_model(**inputs)

test_res = dino_processor\
    .post_process_grounded_object_detection(
        outputs,
        inputs.input_ids,
        target_sizes=[test_img.shape[:2]],
        text_threshold=CRATTT_PARAMS["dino_text_thr"]
    )[0]

print(f"✅ Forward pass OK — {len(test_res['boxes'])} detections")
print(f"   (B=0 init: output identical to pre-LoRA)")

# --- 14.6 VRAM Check ---
vram_used  = torch.cuda.memory_allocated() / 1e9
vram_total = torch.cuda.get_device_properties(0).total_memory / 1e9
vram_free  = vram_total - vram_used

print(f"\n--- VRAM After LoRA Injection ---")
print(f"Used  : {vram_used:.2f} GB")
print(f"Free  : {vram_free:.2f} GB")
print(f"Total : {vram_total:.2f} GB")

if vram_free < 3.0:
    print("⚠️  Less than 3GB free — monitor carefully during backprop")
else:
    print("✅ Sufficient VRAM for TTT backpropagation")

# --- 14.7 Save LoRA Configuration ---
lora_config = {
    "rank":             LORA_RANK,
    "alpha":            LORA_ALPHA,
    "scale":            LORA_ALPHA / LORA_RANK,
    "target_modules":   ["query", "value"],
    "target_scope":     "model.encoder + model.decoder only",
    "n_injected":       n_injected,
    "n_lora_params":    len(lora_params),
    "trainable_params": trainable_params,
    "total_params":     total_params,
    "trainable_ratio":  round(
        100 * trainable_params / total_params, 6
    ),
    "rank4_baseline_params": 73728,
    "capacity_multiplier": round(trainable_params / 73728, 1),
    "reference":        "Hu et al. (2021) LoRA"
}

lora_path = os.path.join(
    EVAL_PARAMS["save_dir"], "lora_config_rank16.json"
)
with open(lora_path, "w") as f:
    json.dump(lora_config, f, indent=2)

print(f"\n✅ LoRA config saved: {lora_path}")
print("\n" + "="*50)
print("BLOCK 14 COMPLETE — LoRA rank=4 injection verified")
print("="*50)

Injecting LoRA adapters into GroundingDINO...
Rank      : 4  (was 4)
Alpha     : 8  (was 8)
Scale     : 2.00  (unchanged — same as rank=4 design)
Targets   : query, value (encoder + decoder only)

✅ LoRA layers injected : 36
✅ Trainable LoRA params: 72

--- Parameter Audit ---
Total parameters    : 172,322,818
Trainable (LoRA)    : 73,728
Frozen (backbone)   : 172,249,090
Trainable ratio     : 0.0428%
vs rank=4 baseline  : 1.0× more capacity
✅ Parameter audit passed

Verifying forward pass after LoRA injection...
✅ Forward pass OK — 31 detections
   (B=0 init: output identical to pre-LoRA)

--- VRAM After LoRA Injection ---
Used  : 2.21 GB
Free  : 13.43 GB
Total : 15.64 GB
✅ Sufficient VRAM for TTT backpropagation

✅ LoRA config saved: /kaggle/working/results/lora_config_rank16.json

BLOCK 14 COMPLETE — LoRA rank=4 injection verified


In [7]:
n_trainable = sum(p.numel() for p in dino_model.parameters() if p.requires_grad)
n_lora_layers = sum(1 for n, p in dino_model.named_parameters() if "lora_B" in n)
print(f"LoRA layers : {n_lora_layers}  (should be 36)")
print(f"Trainable params: {n_trainable:,}  (should be 73,728)")

LoRA layers : 36  (should be 36)
Trainable params: 73,728  (should be 73,728)


In [11]:
def run_dino_with_categories_raw(image_np, box_threshold=0.25):
    dino_model.eval()
    with torch.no_grad():
        inputs = dino_processor(images=image_np, text=DINO_TEXT_PROMPT, return_tensors="pt").to(device)
        outputs = dino_model(**inputs)

    cat_scores_raw, cat_dist = get_category_distribution(outputs, category_token_spans, category_order)
    max_raw_scores, max_cat_idx = cat_scores_raw.max(dim=-1)

    keep_mask    = max_raw_scores >= box_threshold
    keep_indices = keep_mask.nonzero(as_tuple=True)[0]

    if len(keep_indices) == 0:
        return [], [], [], [], None

    img_h, img_w = image_np.shape[:2]
    pred_cxcywh = outputs.pred_boxes[0]
    cx = pred_cxcywh[:, 0] * img_w
    cy = pred_cxcywh[:, 1] * img_h
    pw = pred_cxcywh[:, 2] * img_w
    ph = pred_cxcywh[:, 3] * img_h
    pred_xyxy = torch.stack([cx - pw/2, cy - ph/2, cx + pw/2, cy + ph/2], dim=-1)

    boxes     = pred_xyxy[keep_indices]
    scores    = max_raw_scores[keep_indices]
    labels    = [category_order[i] for i in max_cat_idx[keep_indices].tolist()]
    query_idx = keep_indices.tolist()
    full_dist = cat_dist[keep_indices]

    return boxes, scores, labels, query_idx, full_dist

print("✅ run_dino_with_categories_raw defined")

✅ run_dino_with_categories_raw defined


In [12]:
# ============================================================
# RQ2 Diagnostic: CoT-PL-Inspired Proxy vs Entropy Baseline
# Runtime estimate: ~3-5 minutes (N=15, single corruption/severity)
# ============================================================
import torch
import numpy as np
import pandas as pd
from PIL import Image as PILImage
from torchvision.ops import box_iou
from imagecorruptions import corrupt as ic_corrupt
import random as _py_random
from tqdm.notebook import tqdm
from sklearn.metrics import roc_auc_score
import os

print("=" * 65)
print("RQ2 Diagnostic: Structured 3-Step Proxy vs Entropy Baseline")
print("=" * 65)
print()

required = ["category_order", "category_token_spans", "run_dino_with_categories_raw",
            "clip_model", "clip_processor", "coco_gt", "COCO_MAP"]
missing = [f for f in required if f not in globals()]
if missing:
    raise RuntimeError(f"Missing: {missing}\nRe-run Step 1 (category-distribution extraction) first.")

# ── Corruption seeding (self-contained) ──
CORRUPTION_SEED_OFFSET = {"gaussian_noise": 1000, "motion_blur": 2000, "snow": 3000, "contrast": 4000}
def seed_for_corruption(img_id, corruption, severity):
    offset = CORRUPTION_SEED_OFFSET.get(corruption, 9000)
    return (img_id * 100 + offset + severity) % (2**31)
def apply_corruption_deterministic(raw_img, img_id, corruption, severity):
    seed_val = seed_for_corruption(img_id, corruption, severity)
    np.random.seed(seed_val); _py_random.seed(seed_val)
    return ic_corrupt(raw_img, corruption_name=corruption, severity=severity)

# ── Ground-truth matching (self-contained) ──
def match_detections_to_gt(boxes, labels, img_id, iou_thresh=0.5):
    if len(boxes) == 0:
        return []
    results = [False] * len(boxes)
    claimed_gt = set()
    for cat in set(labels):
        cat_id = COCO_MAP.get(cat)
        if cat_id is None:
            continue
        anns = coco_gt.loadAnns(coco_gt.getAnnIds(imgIds=[img_id], catIds=[cat_id]))
        if not anns:
            continue
        gt_boxes = torch.tensor([
            [a["bbox"][0], a["bbox"][1], a["bbox"][0]+a["bbox"][2], a["bbox"][1]+a["bbox"][3]]
            for a in anns], dtype=torch.float32, device=boxes.device)
        for i in [i for i, l in enumerate(labels) if l == cat]:
            ious = box_iou(boxes[i].unsqueeze(0), gt_boxes)[0]
            best_iou, best_j = ious.max(0)
            key = (cat, best_j.item())
            if best_iou.item() >= iou_thresh and key not in claimed_gt:
                results[i] = True
                claimed_gt.add(key)
    return results

# ── Crop with padding ──
def crop_with_padding(image_np, box, pad_ratio=0.15):
    h, w = image_np.shape[:2]
    x1, y1, x2, y2 = box.tolist()
    bw, bh = x2 - x1, y2 - y1
    x1p, y1p = max(0, x1 - bw*pad_ratio), max(0, y1 - bh*pad_ratio)
    x2p, y2p = min(w, x2 + bw*pad_ratio), min(h, y2 + bh*pad_ratio)
    if x2p <= x1p or y2p <= y1p:
        return None
    crop = image_np[int(y1p):int(y2p), int(x1p):int(x2p)]
    return PILImage.fromarray(crop.astype(np.uint8)) if crop.size > 0 else None

# ── Step 2+3: Recognize (CLIP category sim) + Ground (vs background) ──
def clip_recognize_and_ground(crop_pil, category):
    inputs = clip_processor(
        text=[f"a photo of a {category}", "a photo of background"],
        images=crop_pil, return_tensors="pt", padding=True
    ).to(device)
    with torch.no_grad():
        outputs = clip_model(**inputs)
    logits = outputs.logits_per_image[0]
    return logits[0].item(), logits[1].item()  # cat_sim, bg_sim

def entropy_of_dist(dist_row, eps=1e-8):
    p = dist_row.clamp(min=eps)
    return -(p * torch.log(p)).sum().item()

# ── Main diagnostic loop ──
N_EVAL = 15
CORRUPTION = "motion_blur"
SEVERITY = 5

records = []
for img_path in tqdm(image_files[:N_EVAL], desc=f"{CORRUPTION} sev{SEVERITY}"):
    img_id = img_id_map[os.path.basename(img_path)]
    raw_img = loaded_images[img_path]
    c_img = apply_corruption_deterministic(raw_img, img_id, CORRUPTION, SEVERITY)

    boxes, scores, labels, qidx, dist = run_dino_with_categories_raw(c_img)
    if len(boxes) == 0:
        continue

    correctness = match_detections_to_gt(boxes, labels, img_id)

    for i in range(len(boxes)):
        crop = crop_with_padding(c_img, boxes[i])
        if crop is None:
            continue
        cat_sim, bg_sim = clip_recognize_and_ground(crop, labels[i])
        ent = entropy_of_dist(dist[i])

        records.append({
            "img_id": img_id, "label": labels[i], "correct": correctness[i],
            "cat_sim": cat_sim, "bg_sim": bg_sim, "margin": cat_sim - bg_sim,
            "entropy": ent,
        })

df_rq2 = pd.DataFrame(records)
print(f"\nTotal detections analyzed: {len(df_rq2)}")
print()

# ── Sanity check: eyeball a few examples ──
print("=" * 65)
print("SANITY CHECK — first 8 detections")
print("=" * 65)
print(df_rq2[["label", "correct", "cat_sim", "bg_sim", "margin", "entropy"]].head(8).round(3))
print()

# ── AUC comparison: does each signal discriminate correctness? ──
print("=" * 65)
print("DISCRIMINATION — AUC (continuous signal vs correctness)")
print("=" * 65)
auc_proxy = roc_auc_score(df_rq2["correct"], df_rq2["margin"])
auc_baseline = roc_auc_score(df_rq2["correct"], -df_rq2["entropy"])  # lower entropy = better
print(f"Structured proxy (margin)     AUC: {auc_proxy:.3f}")
print(f"Entropy baseline (-entropy)   AUC: {auc_baseline:.3f}")
print()

# ── Equal-rate binary comparison (median split, fair comparison) ──
print("=" * 65)
print("EQUAL-RATE COMPARISON — median split per method")
print("=" * 65)
df_rq2["proxy_verified"] = df_rq2["margin"] >= df_rq2["margin"].median()
df_rq2["baseline_verified"] = df_rq2["entropy"] <= df_rq2["entropy"].median()

for method, col in [("Structured proxy", "proxy_verified"), ("Entropy baseline", "baseline_verified")]:
    v_prec = df_rq2[df_rq2[col]]["correct"].mean()
    r_prec = df_rq2[~df_rq2[col]]["correct"].mean()
    print(f"{method:20s}  verified={v_prec:.3f}  rejected={r_prec:.3f}  gap={v_prec-r_prec:+.3f}")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_rq2.to_csv("/kaggle/working/tables/table_rq2_diagnostic.csv", index=False)
print("\n✅ Saved: table_rq2_diagnostic.csv")

RQ2 Diagnostic: Structured 3-Step Proxy vs Entropy Baseline



motion_blur sev5:   0%|          | 0/15 [00:00<?, ?it/s]


Total detections analyzed: 57

SANITY CHECK — first 8 detections
          label  correct  cat_sim  bg_sim  margin  entropy
0            tv     True   26.887  24.922   1.965    4.381
1         train    False   21.977  24.981  -3.004    4.381
2  dining table    False   24.201  25.127  -0.927    4.381
3          bear     True   30.956  24.482   6.474    4.377
4  potted plant     True   24.436  25.189  -0.753    4.377
5  dining table    False   24.179  25.617  -1.437    4.381
6          vase    False   24.037  23.802   0.235    4.381
7        person    False   25.691  25.227   0.465    4.381

DISCRIMINATION — AUC (continuous signal vs correctness)
Structured proxy (margin)     AUC: 0.612
Entropy baseline (-entropy)   AUC: 0.783

EQUAL-RATE COMPARISON — median split per method
Structured proxy      verified=0.724  rejected=0.429  gap=+0.296
Entropy baseline      verified=0.793  rejected=0.357  gap=+0.436

✅ Saved: table_rq2_diagnostic.csv


In [13]:
print(df_rq2[["cat_sim", "bg_sim", "margin"]].describe())
print()
print("Correlation: margin vs correct —", df_rq2["margin"].corr(df_rq2["correct"].astype(int)))
print("Correlation: cat_sim vs correct —", df_rq2["cat_sim"].corr(df_rq2["correct"].astype(int)))

         cat_sim     bg_sim     margin
count  57.000000  57.000000  57.000000
mean   26.126058  24.979576   1.146482
std     2.034917   0.813969   2.094568
min    21.977070  23.057220  -3.003988
25%    25.202040  24.481539   0.180674
50%    26.129910  25.051767   0.827885
75%    26.886982  25.567532   2.025005
max    33.673817  26.689482   9.566822

Correlation: margin vs correct — 0.2573549756565671
Correlation: cat_sim vs correct — 0.2537239428010977


In [14]:
# ============================================================
# RQ2 Diagnostic — Gentle Corruption Check: Contrast, Severity 1
# Runtime estimate: <1 minute
# ============================================================
N_EVAL = 15
CORRUPTION = "contrast"
SEVERITY = 1

records_gentle = []
for img_path in tqdm(image_files[:N_EVAL], desc=f"{CORRUPTION} sev{SEVERITY}"):
    img_id = img_id_map[os.path.basename(img_path)]
    raw_img = loaded_images[img_path]
    c_img = apply_corruption_deterministic(raw_img, img_id, CORRUPTION, SEVERITY)

    boxes, scores, labels, qidx, dist = run_dino_with_categories_raw(c_img)
    if len(boxes) == 0:
        continue

    correctness = match_detections_to_gt(boxes, labels, img_id)

    for i in range(len(boxes)):
        crop = crop_with_padding(c_img, boxes[i])
        if crop is None:
            continue
        cat_sim, bg_sim = clip_recognize_and_ground(crop, labels[i])
        ent = entropy_of_dist(dist[i])

        records_gentle.append({
            "img_id": img_id, "label": labels[i], "correct": correctness[i],
            "cat_sim": cat_sim, "bg_sim": bg_sim, "margin": cat_sim - bg_sim,
            "entropy": ent,
        })

df_gentle = pd.DataFrame(records_gentle)
print(f"\nTotal detections analyzed: {len(df_gentle)}")
print()

print("=" * 65)
print("SANITY CHECK — first 8 detections (contrast, severity 1)")
print("=" * 65)
print(df_gentle[["label", "correct", "cat_sim", "bg_sim", "margin", "entropy"]].head(8).round(3))
print()

print("=" * 65)
print("DISCRIMINATION — AUC")
print("=" * 65)
auc_proxy_g = roc_auc_score(df_gentle["correct"], df_gentle["margin"])
auc_baseline_g = roc_auc_score(df_gentle["correct"], -df_gentle["entropy"])
print(f"Structured proxy (margin)     AUC: {auc_proxy_g:.3f}   (motion_blur sev5 was: 0.612)")
print(f"Entropy baseline (-entropy)   AUC: {auc_baseline_g:.3f}   (motion_blur sev5 was: 0.783)")
print()

print("Correlation: margin vs correct  —", df_gentle["margin"].corr(df_gentle["correct"].astype(int)))
print("Correlation: cat_sim vs correct —", df_gentle["cat_sim"].corr(df_gentle["correct"].astype(int)))
print()

print("=" * 65)
print("EQUAL-RATE COMPARISON — median split per method")
print("=" * 65)
df_gentle["proxy_verified"] = df_gentle["margin"] >= df_gentle["margin"].median()
df_gentle["baseline_verified"] = df_gentle["entropy"] <= df_gentle["entropy"].median()

for method, col in [("Structured proxy", "proxy_verified"), ("Entropy baseline", "baseline_verified")]:
    v_prec = df_gentle[df_gentle[col]]["correct"].mean()
    r_prec = df_gentle[~df_gentle[col]]["correct"].mean()
    print(f"{method:20s}  verified={v_prec:.3f}  rejected={r_prec:.3f}  gap={v_prec-r_prec:+.3f}")

os.makedirs("/kaggle/working/tables", exist_ok=True)
df_gentle.to_csv("/kaggle/working/tables/table_rq2_gentle.csv", index=False)
print("\n✅ Saved: table_rq2_gentle.csv")

contrast sev1:   0%|          | 0/15 [00:00<?, ?it/s]


Total detections analyzed: 175

SANITY CHECK — first 8 detections (contrast, severity 1)
          label  correct  cat_sim  bg_sim  margin  entropy
0            tv     True   30.021  24.396   5.625    4.374
1         clock     True   24.347  24.555  -0.208    4.377
2          vase     True   29.175  25.510   3.666    4.378
3         chair     True   24.408  23.756   0.652    4.380
4         chair     True   26.539  24.200   2.340    4.380
5        person     True   24.564  24.005   0.559    4.375
6  potted plant    False   28.649  24.730   3.919    4.379
7     microwave    False   26.555  23.931   2.625    4.381

DISCRIMINATION — AUC
Structured proxy (margin)     AUC: 0.744   (motion_blur sev5 was: 0.612)
Entropy baseline (-entropy)   AUC: 0.876   (motion_blur sev5 was: 0.783)

Correlation: margin vs correct  — 0.38417124437068334
Correlation: cat_sim vs correct — 0.38890345817383204

EQUAL-RATE COMPARISON — median split per method
Structured proxy      verified=0.591  rejected=0.207 

In [ ]:
import time
import torch
import numpy as np

def time_adaptation_step(image_np, params_to_train, n_steps=5, lr=1e-4, n_repeats=10, warmup=2):
    inputs = dino_processor(images=image_np, text=DINO_TEXT_PROMPT, return_tensors="pt").to(device)
    opt = torch.optim.AdamW(params_to_train, lr=lr)

    def run_steps():
        dino_model.train()
        for _ in range(n_steps):
            opt.zero_grad()
            outputs = dino_model(**inputs)
            loss = -outputs.logits.sigmoid().max(dim=-1).values.mean()
            loss.backward()
            opt.step()
        dino_model.eval()

    for _ in range(warmup):
        run_steps()
    torch.cuda.synchronize()

    times = []
    for _ in range(n_repeats):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        run_steps()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return times

print("=" * 65)
print("RQ3 Re-check: LoRA TTT Timing at rank=4")
print("=" * 65)
print()

test_img = loaded_images[image_files[0]]
lora_params = [p for p in dino_model.parameters() if p.requires_grad]
n_lora = sum(p.numel() for p in lora_params)
print(f"LoRA trainable params: {n_lora:,}  (rank=4)")

lora_times_r4 = time_adaptation_step(test_img, lora_params)
print(f"LoRA TTT — mean per-image (5 steps): {np.mean(lora_times_r4)*1000:.1f}ms "
      f"(±{np.std(lora_times_r4)*1000:.1f}ms), per-step: {np.mean(lora_times_r4)/5*1000:.1f}ms")

print()
print("─" * 65)
print(f"Comparison to rank=16 result: 5888.8ms total, 1177.8ms/step")
print(f"Reference: 30 FPS real-time benchmark = 33.3ms/frame")
print("─" * 65)

In [ ]:
import time
import torch

def time_adaptation_step(image_np, params_to_train, n_steps=5, lr=1e-4, n_repeats=10, warmup=2):
    inputs = dino_processor(images=image_np, text=DINO_TEXT_PROMPT, return_tensors="pt").to(device)
    opt = torch.optim.AdamW(params_to_train, lr=lr)

    def run_steps():
        dino_model.train()
        for _ in range(n_steps):
            opt.zero_grad()
            outputs = dino_model(**inputs)
            loss = -outputs.logits.sigmoid().max(dim=-1).values.mean()  # timing proxy loss only
            loss.backward()
            opt.step()
        dino_model.eval()

    for _ in range(warmup):
        run_steps()
    torch.cuda.synchronize()

    times = []
    for _ in range(n_repeats):
        torch.cuda.synchronize()
        t0 = time.perf_counter()
        run_steps()
        torch.cuda.synchronize()
        times.append(time.perf_counter() - t0)
    return times

print("=" * 65)
print("RQ3: Adaptation Speed — LoRA TTT vs Full Fine-Tuning")
print("=" * 65)
print()

test_img = loaded_images[image_files[0]]

# ── Arm 1: LoRA-only (current state — already the only trainable params) ──
lora_params = [p for p in dino_model.parameters() if p.requires_grad]
n_lora = sum(p.numel() for p in lora_params)
print(f"LoRA trainable params: {n_lora:,}")

lora_times = time_adaptation_step(test_img, lora_params)
print(f"LoRA TTT — mean per-image (5 steps): {np.mean(lora_times)*1000:.1f}ms "
      f"(±{np.std(lora_times)*1000:.1f}ms), per-step: {np.mean(lora_times)/5*1000:.1f}ms")
print()

# ── Arm 2: Full fine-tuning — temporarily unfreeze everything ──
original_grad_state = {name: p.requires_grad for name, p in dino_model.named_parameters()}

try:
    for p in dino_model.parameters():
        p.requires_grad = True
    full_params = list(dino_model.parameters())
    n_full = sum(p.numel() for p in full_params)
    print(f"Full fine-tune trainable params: {n_full:,}  ({n_full/n_lora:.0f}x more than LoRA)")

    full_times = time_adaptation_step(test_img, full_params)
    print(f"Full FT — mean per-image (5 steps): {np.mean(full_times)*1000:.1f}ms "
          f"(±{np.std(full_times)*1000:.1f}ms), per-step: {np.mean(full_times)/5*1000:.1f}ms")

    speedup = np.mean(full_times) / np.mean(lora_times)
    print(f"\nSpeedup factor (LoRA vs full FT): {speedup:.2f}x")

except torch.cuda.OutOfMemoryError:
    print("⚠️  Full fine-tuning OOM'd on this GPU — cannot complete this comparison arm as designed.")
    print("   This itself is a relevant finding: full fine-tuning may be infeasible on")
    print("   deployment-class hardware (T4, 15.6GB) where LoRA succeeds.")
    torch.cuda.empty_cache()

finally:
    # Restore exact original frozen state
    for name, p in dino_model.named_parameters():
        p.requires_grad = original_grad_state[name]
    print("\n✅ Original frozen/LoRA state restored")

print()
print("─" * 65)
print("Reference: 30 FPS real-time benchmark = 33.3ms/frame")
print("─" * 65)